# CineRank: Offline Model Training and Release Pipeline (Google Colab)

This notebook is an **offline** training environment. It does not import the backend application, does not connect to any database, does not modify schemas, and does not export production records. The backend database stays authoritative for users, titles, ratings, interactions and watchlists; the backend resolves the MovieLens provider ids in `item_provider_ids.npy` to permanent CineRank title UUIDs itself.

## What this revision changes (relative to the previous notebook)

| Area | Previous behaviour | This notebook |
|---|---|---|
| Feedback | Every rating became a positive implicit event | Ratings `>= POSITIVE_RATING_THRESHOLD` (3.5) are positive implicit events; lower ratings are **not** positives. Explicit ratings stay available (exclusion sets, popularity quality term, explicit-bias diagnostic) |
| Split | One latest-rating holdout, no validation set, movies filtered by sampled counts | Per-user temporal leave-two-out: validation = second-latest positive, test = latest positive. Full catalogue kept; low-frequency items are not removed |
| Evaluation | None | Common users, history, exclusions and candidate sets for every model; Recall/Precision/NDCG/MAP/MRR@5,10,20, coverage, diversity, novelty, long-tail exposure, by cohort and by target-popularity band |
| Popularity | Sum of confidence | Positive-feedback, time-decayed, Bayesian-smoothed, fixed cutoff, deterministic |
| Content | sklearn TF-IDF pickled in memory, weak features | Own reproducible tokenizer, vocabulary + IDF + block weights exported, new items can be vectorised without the training matrix |
| ItemKNN | Cosine top-K | Positive interactions only, optional user down-weighting, shrinkage, validation-selected K |
| ALS | Exported MovieLens user factors | Item factors only; regularised user fold-in is implemented, validated, and used for evaluation |
| BPR | Exported MovieLens user factors | Item factors only; pairwise fold-in implemented; **stays disabled in production unless a validation gate passes** |
| Attention | Small DIN-style scorer, `torch.save`, fixed epochs | Multi-head target-aware attention with positional embeddings, gated fusion, MLP head, pairwise loss, AdamW, AMP, **600 s wall-clock budget**, validation checkpointing, Safetensors + ONNX |
| Hybrid | Hand-set weights | Normalised scores, validation-tuned weights, MMR with an explicit relevance/diversity trade-off, fallbacks |
| Export | Flat bundle with raw ratings, user ids, factors | Versioned release directory with manifest and SHA-256 sums; no raw ratings, user ids, user factors or holdout rows |

## What this notebook does **not** establish

* Ten minutes of attention training is a *compute budget*, not evidence of quality. Training duration, training data volume, optimizer steps, sampled validation metrics and full recommendation metrics are reported separately.
* A sample of MovieLens users is a Colab experiment, not the CineRank population.
* SHA-256 checksums detect corruption; they do **not** authenticate the publisher. Production must verify a trusted signature over `manifest.json` (see `backend/docs/operations/model_deployment.md`).

## Cell types

Cells tagged `config` and `defs` contain only constants, imports and function/class definitions (they do no heavy work). Everything else executes the pipeline. Run top to bottom.

## 1. Install and verify dependencies

In [ ]:
import importlib.util, shutil, subprocess, sys

def _pip(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

HAS_NVIDIA_GPU = shutil.which("nvidia-smi") is not None

# Install each package once. torch is expected to be preinstalled (Colab); install only if missing.
_pip("numpy>=1.26", "pandas>=2.1", "scipy>=1.11", "implicit>=0.7", "safetensors>=0.4",
     "onnx>=1.16", "threadpoolctl>=3.2", "psutil>=5.9", "tqdm>=4.66", "requests>=2.31")
if importlib.util.find_spec("torch") is None:
    _pip("torch>=2.2")

# Exactly one ONNX Runtime distribution must be installed: the GPU build when an NVIDIA GPU exists.
_ort_ok = False
if importlib.util.find_spec("onnxruntime") is not None:
    import onnxruntime as _ort
    _ort_ok = (not HAS_NVIDIA_GPU) or ("CUDAExecutionProvider" in _ort.get_available_providers())
if not _ort_ok:
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime", "onnxruntime-gpu"])
    _pip("onnxruntime-gpu>=1.18" if HAS_NVIDIA_GPU else "onnxruntime>=1.18")
    print("ONNX Runtime was (re)installed. If imports below fail, use Runtime > Restart session and run again.")
print("dependency installation finished; GPU present:", HAS_NVIDIA_GPU)

In [ ]:
# Fail clearly if a required dependency is unavailable.
import importlib
_missing = []
for _name in ["numpy", "pandas", "scipy", "torch", "implicit", "safetensors", "onnx",
              "onnxruntime", "threadpoolctl", "psutil", "tqdm", "requests"]:
    try:
        importlib.import_module(_name)
    except Exception as exc:  # noqa: BLE001
        _missing.append(f"{_name}: {type(exc).__name__}: {exc}")
if _missing:
    raise ImportError("Required dependencies are unavailable:\n  " + "\n  ".join(_missing))
print("all required dependencies import successfully")

## 2. Configuration

All tunable values live in this cell. Nothing here is claimed to be optimal; values that are selected by validation are chosen later and recorded in the run directory.

**Feedback semantics**

| Signal | Definition | Used for |
|---|---|---|
| Explicit rating | The observed 0.5-5.0 rating | Exclusion sets, popularity quality term, explicit-bias diagnostic. Never discarded |
| Positive implicit interaction | rating `>= POSITIVE_RATING_THRESHOLD` | ALS / BPR / ItemKNN / content profile / attention history and targets |
| Negative feedback | rating `<= NEGATIVE_RATING_THRESHOLD` | Counted in the popularity quality denominator; reported; never positive |
| Neutral | between the two thresholds | Not positive; counted as an exposure |
| Missing | no rating | **Not** a confirmed dislike. Sampled "negatives" in BPR/attention are unobserved items, not verified dislikes |

In [ ]:
import os, sys, gc, io, json, time, math, random, shutil, hashlib, zipfile, platform, subprocess, uuid, types, re
import resource, warnings
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import scipy
import scipy.sparse as sp
from tqdm.auto import tqdm

import torch
from torch import nn
import torch.nn.functional as F

try:
    from IPython.display import display
except ImportError:  # running as a plain script
    display = print

# ----- reproducibility and sampling ---------------------------------------------------------
SEED = 42
MOVIELENS_URL = os.environ.get("CINERANK_MOVIELENS_URL", "https://files.grouplens.org/datasets/movielens/ml-latest.zip")
TARGET_USERS = None if os.environ.get("CINERANK_TARGET_USERS", "10000").strip().lower() in {"", "none", "all"} \
    else int(os.environ.get("CINERANK_TARGET_USERS", "10000"))
MIN_USER_RATINGS = 5                    # users with fewer ratings are not in the sampling frame
RATINGS_CHUNK_SIZE = 1_000_000

# ----- feedback semantics ---------------------------------------------------------------------
POSITIVE_RATING_THRESHOLD = 3.5
NEGATIVE_RATING_THRESHOLD = 2.5
IMPLICIT_CONFIDENCE_ALPHA = 40.0        # c_ui = 1 + alpha * relevance(rating), positives only
MIN_POSITIVES_FOR_EVAL = 3              # >=1 train positive + validation target + test target

# ----- evaluation -------------------------------------------------------------------------------
KS = (5, 10, 20)
KMAX = max(KS)
COHORT_EDGES = (1, 5, 20, 100)          # by number of TRAIN positives: new [1,5) low [5,20) medium [20,100) high [100,inf)
COHORT_NAMES = ("new (1-4)", "low (5-19)", "medium (20-99)", "high (100+)")
ITEM_BAND_EDGES = (1, 11, 101)          # by train positives of the TARGET item: unseen 0, tail 1-10, mid 11-100, head 101+
ITEM_BAND_NAMES = ("unseen_in_train", "long_tail", "mid", "head")
EVAL_USERS_PER_COHORT = 1000
TUNE_USERS = 1500                       # validation subset used for per-model hyperparameter grids
EVAL_CHUNK = 64
CANDIDATE_POOL_PER_SOURCE = 100         # top-N from each generator forms the shared candidate pool
COLD_ITEM_POLICY_MODELS = ("attention",)   # score set to pool minimum where popularity_scores == 0

# ----- popularity ---------------------------------------------------------------------------------
POP_HALF_LIFE_DAYS = 365.0
POP_PRIOR_STRENGTH = 10.0

# ----- content ------------------------------------------------------------------------------------
CONTENT_MAX_FEATURES = 30_000
CONTENT_MIN_DF = {"t": 2, "tb": 2, "g": 1, "d": 1}
CONTENT_BLOCK_WEIGHTS = {"t": 1.0, "tb": 0.7, "g": 1.5, "d": 0.5}

# ----- ItemKNN ------------------------------------------------------------------------------------
KNN_K_MAX = 100
KNN_K_GRID = (20, 50, 100)
KNN_WEIGHTINGS = ("binary", "user_idf")
KNN_SHRINKAGE_GRID = (10.0, 50.0)
KNN_MIN_ITEM_SUPPORT = 2
KNN_ROW_BLOCK = 256

# ----- ALS (implicit) -----------------------------------------------------------------------------
ALS_FACTORS_GRID = (32, 64, 128)
ALS_REG_GRID = (0.05, 0.5)
ALS_ITERATIONS_GRID = (15,)

# ----- BPR (implicit) -----------------------------------------------------------------------------
BPR_FACTORS = 64
BPR_LR_GRID = (0.005, 0.02)
BPR_ITERATIONS = 100
BPR_REG = 0.01
BPR_FOLDIN_GRID = ({"learning_rate": 0.03}, {"learning_rate": 0.1})
BPR_FOLDIN_FIXED = {"iterations": 100, "regularization": 0.01, "negatives_per_positive": 3, "max_positives": 100, "seed": SEED}
BPR_USERS_FOR_TRAINED_COMPARISON = 100
BPR_MIN_RELATIVE_NDCG_VS_POPULARITY = 1.0   # gate: fold-in validation NDCG@10 must reach popularity * this

# ----- attention (the 600 s budget applies ONLY to this model) ---------------------------------------
ATTENTION_TIME_BUDGET_SEC = 600.0
ATTENTION_MAX_OVERRUN_SEC = 5.0         # abort the notebook if the measured wall-clock exceeds budget by more than this
ATTENTION_EMBED_DIM = 64
ATTENTION_HEADS = 4
ATTENTION_HIDDEN_DIM = 128
ATTENTION_DROPOUT = 0.1
ATTENTION_MAX_HISTORY = 64
ATTENTION_BATCH_SIZE = 1024
ATTENTION_NUM_NEGATIVES = 4
ATTENTION_UNIFORM_NEG_FRACTION = 0.5    # remainder: popularity^0.75-weighted negatives
ATTENTION_LR = 2e-3
ATTENTION_WEIGHT_DECAY = 0.01           # not applied to embeddings, biases or norms
ATTENTION_WARMUP_STEPS = 200
ATTENTION_MIN_LR_RATIO = 0.05
ATTENTION_GRAD_CLIP = 1.0
ATTENTION_VALIDATION_INTERVAL_SEC = 45.0
ATTENTION_LOG_EVERY_STEPS = 100
ATTENTION_VAL_USERS = 2000
ATTENTION_VAL_NEGATIVES = 99
ATTENTION_ONNX_OPSET = 17
ATTENTION_ONNX_ATOL_CPU = 1e-4          # fp32 vs fp32; accumulation-order differences are ~1e-6
ATTENTION_ONNX_ATOL_GPU = 2e-3          # ORT CUDA may use TF32 / fused kernels

# ----- hybrid ---------------------------------------------------------------------------------------
HYBRID_NORMALIZATIONS = ("minmax", "zscore", "rank")
HYBRID_WEIGHT_TRIALS = 500
MMR_LAMBDA_GRID = (1.0, 0.9, 0.8, 0.7, 0.6, 0.5)
MMR_DEPTH = 50
MMR_MAX_RELATIVE_NDCG_LOSS = 0.02       # choose the most diverse lambda within this NDCG@10 loss on validation

# ----- release policy ------------------------------------------------------------------------------
ALLOW_PARTIAL_RELEASE = False           # True: failed optional models are marked unavailable instead of failing the run
BENCH_USERS = 100

# ----- paths and identifiers -------------------------------------------------------------------------
WORKSPACE = Path(os.environ.get("CINERANK_WORKSPACE", "/content" if Path("/content").exists() else str(Path.cwd()))).expanduser()
DATA_ROOT = Path(os.environ.get("CINERANK_TRAIN_DATA_DIR", str(WORKSPACE / "cinerank_data"))).expanduser()   # training data only
_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
_uniq = uuid.uuid4().hex[:8]
RELEASE_ID = f"cinerank-{_stamp}-{_uniq}"
RUN_ID = f"run-{_stamp}-{_uniq}"
BUILD_ROOT = Path(os.environ.get("CINERANK_BUILD_DIR", str(WORKSPACE / "cinerank_build"))).expanduser() / RELEASE_ID
ARTIFACTS_ROOT = BUILD_ROOT / "backend" / "artifacts"
RELEASE_DIR = ARTIFACTS_ROOT / "models" / "releases" / RELEASE_ID
RUN_DIR = ARTIFACTS_ROOT / "training-runs" / RUN_ID
BUNDLE_ZIP = WORKSPACE / f"{RELEASE_ID}.zip"
ML_DIR = DATA_ROOT / Path(MOVIELENS_URL).stem
ML_ZIP = DATA_ROOT / Path(MOVIELENS_URL).name

CONFIG_NAMES = [n for n in list(globals()) if n.isupper() and not n.startswith("_")
                and isinstance(globals()[n], (int, float, str, bool, tuple, list, dict, type(None)))]
print(json.dumps({"release_id": RELEASE_ID, "run_id": RUN_ID, "target_users": TARGET_USERS,
                  "attention_budget_sec": ATTENTION_TIME_BUDGET_SEC}, indent=1))

## 3. Helpers: seeds, memory, timing, environment, stage tracking

In [ ]:
from threadpoolctl import threadpool_limits, threadpool_info
import psutil


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def rss_mb():
    return psutil.Process().memory_info().rss / 1e6


def peak_rss_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0   # Linux reports KiB


def gpu_mem_used_mb():
    if shutil.which("nvidia-smi") is None:
        return None
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                      text=True, timeout=10)
        return float(out.strip().splitlines()[0])
    except Exception:  # noqa: BLE001
        return None


def percentiles_ms(samples_ms):
    a = np.asarray(samples_ms, dtype=np.float64)
    return {"p50_ms": float(np.percentile(a, 50)), "p95_ms": float(np.percentile(a, 95)),
            "p99_ms": float(np.percentile(a, 99)), "mean_ms": float(a.mean()), "n": int(a.size)}


def time_calls(fn, n, warmup=3, sync=False):
    for _ in range(warmup):
        fn()
    if sync:
        cuda_sync()
    samples = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        if sync:
            cuda_sync()
        samples.append((time.perf_counter() - t0) * 1000.0)
    return percentiles_ms(samples)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (Path,)):
        return str(o)
    if isinstance(o, (set, frozenset, tuple)):
        return list(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def _clean_nonfinite(o):
    if isinstance(o, float) and not math.isfinite(o):
        return None
    if isinstance(o, dict):
        return {str(k): _clean_nonfinite(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean_nonfinite(v) for v in o]
    return o


def write_json(path, obj, indent=2):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(_clean_nonfinite(obj), indent=indent, default=_json_default, ensure_ascii=False, allow_nan=False)
    path.write_text(text + "\n", encoding="utf-8")


def capture_environment():
    import onnx, onnxruntime, safetensors
    try:
        import implicit
        implicit_version = getattr(implicit, "__version__", "unknown")
    except Exception as exc:  # noqa: BLE001
        implicit_version = f"unavailable: {exc}"
    env = {
        "python": platform.python_version(), "platform": platform.platform(),
        "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__,
        "torch": torch.__version__, "torch_cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "onnx": onnx.__version__, "onnxruntime": onnxruntime.__version__,
        "onnxruntime_available_providers": onnxruntime.get_available_providers(),
        "safetensors": safetensors.__version__, "implicit": implicit_version,
        "cpu_count": os.cpu_count(), "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 2),
        "cuda_available": torch.cuda.is_available(),
        "blas_threadpools": [{k: v for k, v in d.items() if k in ("internal_api", "num_threads", "threading_layer")}
                             for d in threadpool_info()],
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        env.update({"gpu_name": props.name, "gpu_memory_gb": round(props.total_memory / 1e9, 2),
                    "gpu_compute_capability": f"{props.major}.{props.minor}"})
    return env


STAGES = {}          # model name -> {"status": ..., "error": ...}


@contextmanager
def model_stage(name, critical=False):
    """Run a model stage. A failure is recorded; the run is aborted unless the model is optional
    and ALLOW_PARTIAL_RELEASE is True. Critical models (popularity, attention) always abort."""
    t0 = time.perf_counter()
    try:
        yield
        STAGES[name] = {"status": "available", "seconds": round(time.perf_counter() - t0, 2)}
    except Exception as exc:  # noqa: BLE001
        STAGES[name] = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}
        print(f"!! stage '{name}' failed: {type(exc).__name__}: {exc}")
        if critical or not ALLOW_PARTIAL_RELEASE:
            raise
    finally:
        free_memory()

## 4. Reference inference module (single source of truth for the inference contract)

The code below is written to `training-runs/<run_id>/reference_inference.py` and executed here. Every artifact check, fold-in, fusion and end-to-end test in this notebook uses this exact code, so the contract that is documented is the contract that was tested.

In [ ]:
REFERENCE_INFERENCE_SRC = r'''"""CineRank reference inference (NumPy / SciPy; onnxruntime is optional).

This file is the single source of truth for the inference contract. The training
notebook embeds it verbatim, executes it, and tests the release artifacts with it.
The backend may copy it or re-implement it; any re-implementation must match
the behaviour documented in backend/docs/ml/model_artifacts.md.

Rules:
  * No pickle, no joblib. np.load is always called with allow_pickle=False.
  * Item indices are positions in item_provider_ids.npy. The backend maps
    provider ids to permanent CineRank title ids through its own database.
  * All functions are deterministic for identical inputs.
"""
import hashlib
import json
import math
import re
import time
import unicodedata
import zipfile
from pathlib import Path

import numpy as np
import scipy.sparse as sp

SCHEMA_VERSION = 1

MODEL_FILES = {
    "popularity": ["popularity_scores.npy", "popularity_config.json"],
    "content": ["content_item_vectors.npz", "content_idf.npy", "content_config.json"],
    "itemknn": ["itemknn_topk.npz", "itemknn_config.json"],
    "als": ["als_item_factors.npy", "als_config.json"],
    "bpr": ["bpr_item_factors.npy", "bpr_config.json"],
    "attention": ["attention.safetensors", "attention.onnx", "attention_config.json"],
    "hybrid": ["hybrid_config.json"],
}
ITEM_ID_FILE = "item_provider_ids.npy"
ALWAYS_PRESENT = ["manifest.json", "SHA256SUMS", ITEM_ID_FILE]


# --------------------------------------------------------------------------
# Feedback semantics
# --------------------------------------------------------------------------
def rating_to_relevance(rating):
    """Map an explicit rating in [0.5, 5.0] to a strength in [0, 1]."""
    r = np.asarray(rating, dtype=np.float32)
    return np.clip((r - 0.5) / 4.5, 0.0, 1.0).astype(np.float32)


# --------------------------------------------------------------------------
# Deterministic ranking helpers
# --------------------------------------------------------------------------
def topk_indices(scores, k, exclude=None):
    """Top-k indices of a 1-D score vector.

    Order: descending score, ties broken by ascending index. Excluded and
    non-finite entries are never returned. May return fewer than k indices.
    """
    s = np.array(scores, dtype=np.float32, copy=True).ravel()
    s[~np.isfinite(s)] = -np.inf
    if exclude is not None and len(exclude):
        s[np.asarray(exclude, dtype=np.int64)] = -np.inf
    k = int(min(k, s.size))
    if k <= 0:
        return np.zeros(0, dtype=np.int64)
    kth = np.partition(s, s.size - k)[s.size - k]
    greater = np.flatnonzero(s > kth)
    ties = np.flatnonzero(s == kth)[: k - greater.size]
    cand = np.concatenate([greater, ties])
    cand = cand[np.isfinite(s[cand])]
    order = np.lexsort((cand, -s[cand]))
    return cand[order].astype(np.int64)


# --------------------------------------------------------------------------
# Content-based TF-IDF (own tokenizer; no sklearn needed at inference)
# --------------------------------------------------------------------------
CONTENT_STOPWORDS = frozenset(
    "a an and are as at be by for from has he her his in is it its of on or that the "
    "their there this to was were will with".split()
)
_YEAR_RE = re.compile(r"\((\d{4})\)\s*$")
_NON_ALNUM = re.compile(r"[^a-z0-9]+")


def normalize_text(text):
    text = unicodedata.normalize("NFKD", str(text if text is not None else ""))
    return text.encode("ascii", "ignore").decode("ascii").lower()


def split_title_year(title):
    title = str(title if title is not None else "").strip()
    m = _YEAR_RE.search(title)
    if m:
        return title[: m.start()].strip(), int(m.group(1))
    return title, None


def content_tokens(title, genres, year=None):
    """Tokens for one catalogue item. Prefixes: t: title word, tb: title bigram,
    g: genre, d: decade. Output is a sorted list of unique tokens."""
    clean_title, parsed_year = split_title_year(title)
    if year is None:
        year = parsed_year
    words = [w for w in _NON_ALNUM.split(normalize_text(clean_title))
             if w and w not in CONTENT_STOPWORDS and (len(w) > 1 or w.isdigit())]
    tokens = {"t:" + w for w in words}
    tokens.update("tb:" + a + "_" + b for a, b in zip(words, words[1:]))
    if isinstance(genres, str):
        genre_list = genres.split("|")
    else:
        genre_list = list(genres) if genres is not None else []
    for g in genre_list:
        g = _NON_ALNUM.sub("_", normalize_text(g)).strip("_")
        if g and g != "no_genres_listed":
            tokens.add("g:" + g)
    if year is not None and 1800 <= int(year) <= 2100:
        tokens.add("d:%ds" % (int(year) // 10 * 10))
    return sorted(tokens)


def fit_content_vocabulary(token_lists, min_df_by_prefix, max_features):
    df = {}
    for toks in token_lists:
        for t in set(toks):
            df[t] = df.get(t, 0) + 1
    kept = [(t, c) for t, c in df.items() if c >= min_df_by_prefix.get(t.split(":", 1)[0], 1)]
    kept.sort(key=lambda x: (-x[1], x[0]))
    kept = kept[: int(max_features)]
    kept.sort(key=lambda x: x[0])
    vocabulary = {t: i for i, (t, _) in enumerate(kept)}
    n = len(token_lists)
    idf = np.asarray([math.log((1.0 + n) / (1.0 + c)) + 1.0 for _, c in kept], dtype=np.float32)
    return vocabulary, idf


def content_vectors(token_lists, vocabulary, idf, block_weights):
    """L2-normalised sparse TF-IDF rows (binary tf). Same code path for training
    items and for new catalogue items at inference time."""
    indptr = [0]
    indices = []
    data = []
    for toks in token_lists:
        row = {}
        for t in set(toks):
            j = vocabulary.get(t)
            if j is None:
                continue
            row[j] = float(block_weights.get(t.split(":", 1)[0], 1.0)) * float(idf[j])
        cols = sorted(row)
        vals = np.asarray([row[c] for c in cols], dtype=np.float64)
        norm = float(np.linalg.norm(vals)) if vals.size else 0.0
        if norm > 0.0:
            vals = vals / norm
        indices.extend(cols)
        data.extend(vals.tolist())
        indptr.append(len(indices))
    return sp.csr_matrix(
        (np.asarray(data, dtype=np.float32), np.asarray(indices, dtype=np.int32),
         np.asarray(indptr, dtype=np.int64)),
        shape=(len(token_lists), len(vocabulary)),
    )


def history_matrix(histories, weights, n_items):
    """Sparse [B x n_items] matrix; row b holds weights[b] at histories[b]."""
    rows, cols, vals = [], [], []
    for b, (items, w) in enumerate(zip(histories, weights)):
        items = np.asarray(items, dtype=np.int64)
        if items.size == 0:
            continue
        w = np.ones(items.size, dtype=np.float32) if w is None else np.asarray(w, dtype=np.float32)
        rows.append(np.full(items.size, b, dtype=np.int64))
        cols.append(items)
        vals.append(w)
    if not rows:
        return sp.csr_matrix((len(histories), n_items), dtype=np.float32)
    return sp.csr_matrix(
        (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
        shape=(len(histories), n_items), dtype=np.float32,
    )


def _row_normalize(m):
    m = m.tocsr().astype(np.float32)
    norms = np.sqrt(np.asarray(m.multiply(m).sum(axis=1)).ravel())
    inv = np.divide(1.0, norms, out=np.zeros_like(norms), where=norms > 0).astype(np.float32)
    return sp.diags(inv) @ m


def content_scores(item_vectors, hist_matrix):
    """Cosine between each user profile (sum of positive-history item vectors) and all items."""
    profile = _row_normalize(hist_matrix @ item_vectors)
    return np.asarray((profile @ item_vectors.T).todense(), dtype=np.float32)


def itemknn_scores(topk_similarity, hist_matrix):
    return np.asarray((hist_matrix @ topk_similarity).todense(), dtype=np.float32)


# --------------------------------------------------------------------------
# ALS fold-in (regularised least squares with fixed item factors)
# --------------------------------------------------------------------------
def als_gram(item_factors):
    y = np.asarray(item_factors, dtype=np.float64)
    return y.T @ y


def als_fold_in(item_factors, gram, item_indices, relevance, alpha, reg):
    """x_u = (Y^T Y + sum_i (c_ui - 1) y_i y_i^T + reg I)^-1 sum_i c_ui y_i ,
    with c_ui = 1 + alpha * relevance_ui over the user's positive items."""
    idx = np.asarray(item_indices, dtype=np.int64)
    f = item_factors.shape[1]
    if idx.size == 0:
        return np.zeros(f, dtype=np.float32)
    y = np.asarray(item_factors[idx], dtype=np.float64)
    c = 1.0 + float(alpha) * np.asarray(relevance, dtype=np.float64)
    a = gram + (y.T * (c - 1.0)) @ y + float(reg) * np.eye(f)
    b = y.T @ c
    return np.linalg.solve(a, b).astype(np.float32)


# --------------------------------------------------------------------------
# BPR fold-in (item factors fixed; only the user vector is optimised)
# item_factors layout (implicit library): [:, :-1] latent factors, [:, -1] item bias.
# --------------------------------------------------------------------------
def bpr_fold_in(item_factors, positive_items, seen_items, n_iterations=100, learning_rate=0.05,
                reg=0.01, negatives_per_positive=3, seed=0, max_positives=100,
                return_diagnostics=False):
    v = item_factors[:, :-1]
    bias = item_factors[:, -1]
    n_items, f = v.shape
    pos = np.unique(np.asarray(positive_items, dtype=np.int64))
    if pos.size > max_positives:
        pos = pos[-max_positives:]
    if pos.size == 0:
        u0 = np.zeros(f, dtype=np.float32)
        return (u0, {"loss_start": None, "loss_end": None}) if return_diagnostics else u0
    seen = np.unique(np.asarray(seen_items, dtype=np.int64))
    rng = np.random.default_rng(int(seed))
    u = rng.normal(0.0, 0.01, f)
    m = np.zeros(f)
    s = np.zeros(f)
    pos_rep = np.repeat(pos, int(negatives_per_positive))
    vp = np.asarray(v[pos_rep], dtype=np.float64)
    bp = np.asarray(bias[pos_rep], dtype=np.float64)
    losses = []
    for t in range(1, int(n_iterations) + 1):
        neg = rng.integers(0, n_items, size=pos_rep.size)
        for _ in range(10):
            hit = np.isin(neg, seen)
            if not hit.any():
                break
            neg[hit] = rng.integers(0, n_items, size=int(hit.sum()))
        keep = ~np.isin(neg, seen)
        if not keep.any():
            break
        dv = vp[keep] - np.asarray(v[neg[keep]], dtype=np.float64)
        db = bp[keep] - np.asarray(bias[neg[keep]], dtype=np.float64)
        x = dv @ u + db
        losses.append(float(np.mean(np.logaddexp(0.0, -x)) + 0.5 * reg * float(u @ u)))
        g = -1.0 / (1.0 + np.exp(np.clip(x, -50.0, 50.0)))
        grad = (g[:, None] * dv).mean(axis=0) + reg * u
        m = 0.9 * m + 0.1 * grad
        s = 0.999 * s + 0.001 * grad * grad
        u = u - learning_rate * (m / (1 - 0.9 ** t)) / (np.sqrt(s / (1 - 0.999 ** t)) + 1e-8)
    u32 = u.astype(np.float32)
    if return_diagnostics:
        head = float(np.mean(losses[:5])) if losses else None
        tail = float(np.mean(losses[-5:])) if losses else None
        return u32, {"loss_start": head, "loss_end": tail}
    return u32


def bpr_seed(base_seed, positive_items):
    """Deterministic per-history seed so identical histories always fold in identically."""
    return int(base_seed) + int(np.sum(np.asarray(positive_items, dtype=np.int64)) % 1000003)


def bpr_scores(item_factors, user_vectors):
    u = np.atleast_2d(np.asarray(user_vectors, dtype=np.float32))
    return (u @ item_factors[:, :-1].T + item_factors[:, -1][None, :]).astype(np.float32)


# --------------------------------------------------------------------------
# Score normalisation, fusion, diversity
# --------------------------------------------------------------------------
def normalize_scores(scores, valid, method="minmax", rrf_k=60.0, eps=1e-9):
    """Row-wise normalisation of [B x C] scores. Invalid entries map to the row
    minimum of the normalised range. Methods: minmax, zscore, rank, rrf."""
    s = np.asarray(scores, dtype=np.float64)
    valid = np.asarray(valid, dtype=bool)
    s = np.where(valid, s, 0.0)
    has = valid.any(axis=1, keepdims=True)
    if method == "minmax":
        lo = np.where(valid, s, np.inf).min(axis=1, keepdims=True)
        hi = np.where(valid, s, -np.inf).max(axis=1, keepdims=True)
        lo = np.where(has, lo, 0.0)
        hi = np.where(has, hi, 0.0)
        out = (s - lo) / (hi - lo + eps)
        return np.where(valid, out, 0.0).astype(np.float32)
    if method == "zscore":
        n = np.maximum(valid.sum(axis=1, keepdims=True), 1)
        mean = np.where(valid, s, 0.0).sum(axis=1, keepdims=True) / n
        var = np.where(valid, (s - mean) ** 2, 0.0).sum(axis=1, keepdims=True) / n
        z = (s - mean) / np.sqrt(var + eps)
        zmin = np.where(valid, z, np.inf).min(axis=1, keepdims=True)
        zmin = np.where(has, zmin, 0.0)
        return np.where(valid, z, zmin).astype(np.float32)
    if method in ("rank", "rrf"):
        key = np.where(valid, -s, np.inf)
        order = np.argsort(key, axis=1, kind="stable")
        ranks = np.empty_like(order)
        np.put_along_axis(ranks, order, np.arange(s.shape[1])[None, :].repeat(s.shape[0], 0), axis=1)
        if method == "rank":
            nvalid = np.maximum(valid.sum(axis=1, keepdims=True), 1)
            out = 1.0 - ranks / nvalid
        else:
            out = 1.0 / (rrf_k + ranks + 1.0)
        return np.where(valid, out, 0.0).astype(np.float32)
    raise ValueError("unknown normalisation method: %r" % (method,))


def fuse_scores(normalized, weights, row_available=None):
    """Weighted sum of normalised score matrices. A model unavailable for a row
    is dropped for that row and the remaining weights are renormalised."""
    num = None
    den = None
    for name, w in weights.items():
        if name not in normalized or float(w) <= 0.0:
            continue
        mat = np.asarray(normalized[name], dtype=np.float64)
        if row_available is not None and name in row_available:
            a = np.asarray(row_available[name], dtype=np.float64).reshape(-1, 1)
        else:
            a = np.ones((mat.shape[0], 1))
        num = float(w) * a * mat if num is None else num + float(w) * a * mat
        den = float(w) * a if den is None else den + float(w) * a
    if num is None:
        raise ValueError("no model with a positive weight is available for fusion")
    out = np.zeros_like(num)
    np.divide(num, np.broadcast_to(den, num.shape), out=out, where=np.broadcast_to(den > 0, num.shape))
    return out.astype(np.float32)


def mmr_rerank(scores, candidate_items, item_vectors, k, lam, depth=50):
    """Maximal Marginal Relevance over the top `depth` candidates.

    Returns positions into `scores`. lam=1 keeps the relevance order.
    Similarity is content cosine (item_vectors rows are L2-normalised)."""
    scores = np.asarray(scores, dtype=np.float64)
    candidate_items = np.asarray(candidate_items, dtype=np.int64)
    depth = int(min(depth, scores.size))
    top = topk_indices(scores, depth)
    if lam >= 1.0 or item_vectors is None or top.size <= 1:
        return top[:k]
    s = scores[top]
    rel = (s - s.min()) / (s.max() - s.min() + 1e-9)
    vec = item_vectors[candidate_items[top]]
    sim = np.asarray((vec @ vec.T).todense(), dtype=np.float64)
    selected = [0]
    remaining = list(range(1, top.size))
    max_sim = sim[0].copy()
    while len(selected) < min(k, top.size):
        cand = np.asarray(remaining)
        value = lam * rel[cand] - (1.0 - lam) * max_sim[cand]
        j = int(cand[int(np.argmax(value))])
        selected.append(j)
        remaining.remove(j)
        max_sim = np.maximum(max_sim, sim[j])
    return top[np.asarray(selected, dtype=np.int64)]


# --------------------------------------------------------------------------
# Attention helpers (ONNX Runtime)
# --------------------------------------------------------------------------
def pad_histories(histories, max_history):
    """Right-aligned, left-padded int64 [B x max_history]. Ids are item_index + 1; 0 is padding."""
    out = np.zeros((len(histories), max_history), dtype=np.int64)
    for i, h in enumerate(histories):
        h = np.asarray(h, dtype=np.int64)[-max_history:]
        if h.size:
            out[i, max_history - h.size:] = h + 1
    return out


def make_ort_session(onnx_path, use_gpu=False):
    import onnxruntime as ort
    available = ort.get_available_providers()
    if use_gpu and "CUDAExecutionProvider" in available:
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    else:
        providers = ["CPUExecutionProvider"]
    options = ort.SessionOptions()
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(onnx_path), sess_options=options, providers=providers)


def attention_scores_onnx(session, history_ids, candidate_ids):
    out = session.run(["scores"], {
        "history": np.ascontiguousarray(history_ids, dtype=np.int64),
        "candidates": np.ascontiguousarray(candidate_ids, dtype=np.int64),
    })
    return out[0]


# --------------------------------------------------------------------------
# Release integrity and safe loading
# --------------------------------------------------------------------------
def file_sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def write_sha256sums(directory):
    directory = Path(directory)
    lines = []
    for p in sorted(directory.rglob("*")):
        if p.is_file() and p.name != "SHA256SUMS":
            lines.append("%s  %s" % (file_sha256(p), p.relative_to(directory).as_posix()))
    (directory / "SHA256SUMS").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return len(lines)


def verify_sha256sums(directory):
    """Raise ValueError if any listed file is missing/modified or any file is unlisted."""
    directory = Path(directory)
    sums = directory / "SHA256SUMS"
    if not sums.is_file():
        raise ValueError("SHA256SUMS is missing")
    listed = {}
    for line in sums.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        digest, name = line.split("  ", 1)
        listed[name] = digest
    problems = []
    for name, digest in listed.items():
        p = directory / name
        if not p.is_file():
            problems.append("missing file: " + name)
        elif file_sha256(p) != digest:
            problems.append("checksum mismatch: " + name)
    actual = {p.relative_to(directory).as_posix() for p in directory.rglob("*")
              if p.is_file() and p.name != "SHA256SUMS"}
    for extra in sorted(actual - set(listed)):
        problems.append("unlisted file: " + extra)
    if problems:
        raise ValueError("; ".join(problems))
    return len(listed)


def safe_extract_zip(archive, destination, max_total_bytes=8 * 1024 ** 3, max_files=10000):
    """Extract a ZIP without path traversal, absolute paths, symlinks or zip bombs."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as zf:
        infos = zf.infolist()
        if len(infos) > max_files:
            raise ValueError("archive has too many members")
        if sum(i.file_size for i in infos) > max_total_bytes:
            raise ValueError("archive expands beyond the permitted size")
        for info in infos:
            name = info.filename
            if name.startswith(("/", "\\")) or re.match(r"^[A-Za-z]:", name) or ".." in Path(name).parts:
                raise ValueError("unsafe ZIP member path: " + name)
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("symlink in archive: " + name)
            target = (destination / name).resolve()
            if target != root and root not in target.parents:
                raise ValueError("unsafe ZIP member path: " + name)
        for info in infos:
            target = (destination / info.filename).resolve()
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(target, "wb") as dst:
                while True:
                    block = src.read(1 << 20)
                    if not block:
                        break
                    dst.write(block)


def _check_finite(name, arr):
    if not np.all(np.isfinite(arr)):
        raise ValueError("%s contains NaN or infinity" % name)


def _check_csr(name, m, shape, dtype=np.float32):
    if not sp.isspmatrix_csr(m):
        raise ValueError("%s is not CSR" % name)
    if m.shape != tuple(shape):
        raise ValueError("%s has shape %s, expected %s" % (name, m.shape, tuple(shape)))
    if m.dtype != dtype:
        raise ValueError("%s has dtype %s, expected %s" % (name, m.dtype, np.dtype(dtype)))
    _check_finite(name, m.data)
    if m.nnz:
        if m.indices.min() < 0 or m.indices.max() >= shape[1]:
            raise ValueError("%s has out-of-range column indices" % name)
    if np.any(np.diff(m.indptr) < 0) or m.indptr[0] != 0 or m.indptr[-1] != m.nnz:
        raise ValueError("%s has an invalid indptr" % name)


class Release:
    """A loaded, validated model release. Arrays are loaded once and kept in memory."""

    def __init__(self, directory):
        self.directory = Path(directory)
        self.manifest = json.loads((self.directory / "manifest.json").read_text(encoding="utf-8"))
        self.item_provider_ids = np.load(self.directory / ITEM_ID_FILE, allow_pickle=False)
        self.n_items = int(self.item_provider_ids.shape[0])
        self.models = {}
        self.configs = {}
        self.arrays = {}

    def model_available(self, name):
        return self.manifest["models"].get(name, {}).get("status") == "available"

    def model_enabled(self, name):
        entry = self.manifest["models"].get(name, {})
        return entry.get("status") == "available" and bool(entry.get("production_enabled", False))


def load_release(directory, verify_checksums=True, require_complete=True):
    """Load and validate a release directory. Raises ValueError on any problem.
    Production callers use the defaults; the training notebook relaxes both only for a draft release."""
    directory = Path(directory)
    if verify_checksums:
        verify_sha256sums(directory)
    rel = Release(directory)
    man = rel.manifest
    if require_complete and man.get("status") != "complete":
        raise ValueError("release is not marked complete")
    if int(man.get("schema_version", -1)) != SCHEMA_VERSION:
        raise ValueError("unsupported manifest schema_version")
    ids = rel.item_provider_ids
    if ids.ndim != 1 or ids.dtype != np.int64:
        raise ValueError("item_provider_ids must be a 1-D int64 array")
    if np.unique(ids).size != ids.size:
        raise ValueError("item_provider_ids contains duplicates")
    n = rel.n_items
    for name, entry in man["models"].items():
        if entry.get("status") != "available":
            continue
        for fname in MODEL_FILES[name]:
            if not (directory / fname).is_file():
                raise ValueError("%s: missing file %s" % (name, fname))
        if name == "popularity":
            arr = np.load(directory / "popularity_scores.npy", allow_pickle=False)
            if arr.shape != (n,) or arr.dtype != np.float32:
                raise ValueError("popularity_scores shape/dtype mismatch")
            _check_finite("popularity_scores", arr)
            rel.arrays["popularity"] = arr
        elif name == "content":
            cfg = json.loads((directory / "content_config.json").read_text(encoding="utf-8"))
            idf = np.load(directory / "content_idf.npy", allow_pickle=False)
            vec = sp.load_npz(directory / "content_item_vectors.npz").tocsr()
            v = len(cfg["vocabulary"])
            if idf.shape != (v,) or idf.dtype != np.float32:
                raise ValueError("content idf shape/dtype mismatch")
            _check_finite("content_idf", idf)
            _check_csr("content_item_vectors", vec, (n, v))
            rel.arrays["content"] = vec
            rel.arrays["content_idf"] = idf
            rel.configs["content"] = cfg
        elif name == "itemknn":
            cfg = json.loads((directory / "itemknn_config.json").read_text(encoding="utf-8"))
            m = sp.load_npz(directory / "itemknn_topk.npz").tocsr()
            _check_csr("itemknn_topk", m, (n, n))
            if np.any(m.data < 0):
                raise ValueError("itemknn_topk has negative similarities")
            if m.nnz and int(np.diff(m.indptr).max()) > int(cfg["k"]):
                raise ValueError("itemknn_topk has more than k neighbours in a row")
            if m.nnz and np.any(m.indices == np.repeat(np.arange(n), np.diff(m.indptr))):
                raise ValueError("itemknn_topk contains self-similarities")
            rel.arrays["itemknn"] = m
            rel.configs["itemknn"] = cfg
        elif name in ("als", "bpr"):
            cfg = json.loads((directory / (name + "_config.json")).read_text(encoding="utf-8"))
            arr = np.load(directory / (name + "_item_factors.npy"), allow_pickle=False)
            expected = (n, int(cfg["item_factor_columns"]))
            if arr.shape != expected or arr.dtype != np.float32:
                raise ValueError("%s item factors shape/dtype mismatch: %s" % (name, arr.shape))
            _check_finite(name + "_item_factors", arr)
            rel.arrays[name] = arr
            rel.configs[name] = cfg
            if name == "als":
                rel.arrays["als_gram"] = als_gram(arr)
        elif name == "attention":
            cfg = json.loads((directory / "attention_config.json").read_text(encoding="utf-8"))
            if int(cfg["n_items"]) != n:
                raise ValueError("attention n_items does not match item_provider_ids")
            rel.configs["attention"] = cfg
        elif name == "hybrid":
            rel.configs["hybrid"] = json.loads((directory / "hybrid_config.json").read_text(encoding="utf-8"))
    if "hybrid" in rel.configs:
        hc = rel.configs["hybrid"]
        w = hc["weights"]
        if abs(sum(w.values()) - 1.0) > 1e-6:
            raise ValueError("hybrid weights do not sum to 1")
        for m in w:
            if not rel.model_enabled(m):
                raise ValueError("hybrid weights reference unavailable or disabled model: " + m)
    return rel


class CineRankRecommender:
    """Loads a release once and serves recommendations for one user at a time.

    history: sequence of (item_index, rating) ordered oldest -> newest.
    Items are release item indices (positions in item_provider_ids.npy).
    """

    def __init__(self, release_dir, use_gpu=False, disabled_models=(), verify_checksums=True, require_complete=True):
        self.release = load_release(release_dir, verify_checksums=verify_checksums, require_complete=require_complete)
        rel = self.release
        self.hybrid = rel.configs["hybrid"]
        self.disabled = set(disabled_models)
        self.n_items = rel.n_items
        self.popularity = rel.arrays["popularity"]
        self.cold_mask = self.popularity <= 0.0
        self.threshold = float(self.hybrid["feedback"]["positive_rating_threshold"])
        self.session = None
        self.attention_cfg = rel.configs.get("attention")
        if rel.model_enabled("attention") and "attention" not in self.disabled:
            self.session = make_ort_session(rel.directory / "attention.onnx", use_gpu=use_gpu)
        self.attention_provider = self.session.get_providers()[0] if self.session else None

    def _active(self, name):
        return self.release.model_enabled(name) and name not in self.disabled and (
            name != "attention" or self.session is not None)

    def _dense_scores(self, name, pos_items, pos_rel, seen):
        rel = self.release
        n = self.n_items
        if name == "popularity":
            return self.popularity.astype(np.float32)
        if name == "content":
            h = history_matrix([pos_items], [None], n)
            return content_scores(rel.arrays["content"], h)[0]
        if name == "itemknn":
            h = history_matrix([pos_items], [None], n)
            return itemknn_scores(rel.arrays["itemknn"], h)[0]
        if name == "als":
            cfg = rel.configs["als"]
            x = als_fold_in(rel.arrays["als"], rel.arrays["als_gram"], pos_items, pos_rel,
                            cfg["confidence_alpha"], cfg["regularization"])
            return (rel.arrays["als"] @ x).astype(np.float32)
        if name == "bpr":
            cfg = rel.configs["bpr"]
            fi = cfg["fold_in"]
            u = bpr_fold_in(rel.arrays["bpr"], pos_items, seen, fi["iterations"], fi["learning_rate"],
                            fi["regularization"], fi["negatives_per_positive"],
                            bpr_seed(fi["seed"], pos_items), fi["max_positives"])
            return bpr_scores(rel.arrays["bpr"], u)[0]
        raise KeyError(name)

    def recommend(self, history, k=20, exclude=None, return_timings=False):
        t_all = time.perf_counter()
        timings = {}
        n = self.n_items
        items = np.asarray([h[0] for h in history], dtype=np.int64)
        ratings = np.asarray([h[1] for h in history], dtype=np.float32)
        if items.size and (items.min() < 0 or items.max() >= n):
            raise ValueError("history item index out of range")
        # de-duplicate keeping the latest occurrence, preserve chronological order
        if items.size:
            _, last_rev = np.unique(items[::-1], return_index=True)
            keep = np.sort(items.size - 1 - last_rev)
            items, ratings = items[keep], ratings[keep]
        pos_mask = ratings >= self.threshold
        pos_items = items[pos_mask]
        pos_rel = rating_to_relevance(ratings[pos_mask])
        excluded = np.unique(np.concatenate([items, np.asarray(exclude if exclude is not None else [],
                                                                  dtype=np.int64)]))
        used, fallback = [], None

        def finish(idx, scores, tag):
            timings["total_ms"] = (time.perf_counter() - t_all) * 1000.0
            res = {"items": np.asarray(idx, dtype=np.int64), "scores": np.asarray(scores, dtype=np.float32),
                   "models_used": tag, "fallback": fallback}
            if return_timings:
                res["timings_ms"] = timings
            return res

        if pos_items.size == 0:
            fallback = "cold_start_popularity"
            idx = topk_indices(self.popularity, k, excluded)
            return finish(idx, self.popularity[idx], ["popularity"])

        gen_cfg = self.hybrid["candidate_generation"]
        score_models = [m for m in self.hybrid["scoring_models"] if self._active(m)]
        generators = [m for m in gen_cfg["sources"] if self._active(m)]
        if not generators or not score_models:
            fallback = "popularity_only"
            idx = topk_indices(self.popularity, k, excluded)
            return finish(idx, self.popularity[idx], ["popularity"])
        dense = {}
        for name in set(score_models) | set(generators):
            if name == "attention":
                continue
            t0 = time.perf_counter()
            dense[name] = self._dense_scores(name, pos_items, pos_rel, items)
            timings[name + "_ms"] = (time.perf_counter() - t0) * 1000.0
        t0 = time.perf_counter()
        pool = np.unique(np.concatenate(
            [topk_indices(dense[g], int(gen_cfg["per_source"]), excluded) for g in generators]))
        timings["pool_ms"] = (time.perf_counter() - t0) * 1000.0
        if pool.size == 0:
            fallback = "empty_pool_popularity"
            idx = topk_indices(self.popularity, k, excluded)
            return finish(idx, self.popularity[idx], ["popularity"])
        pool_scores = {}
        for name in score_models:
            if name == "attention":
                t0 = time.perf_counter()
                hist = pad_histories([pos_items], self.attention_cfg["max_history"])
                s = attention_scores_onnx(self.session, hist, (pool + 1)[None, :])[0].astype(np.float64)
                cold = self.cold_mask[pool]
                if cold.any() and (~cold).any():
                    s[cold] = s[~cold].min()
                pool_scores[name] = s
                timings["attention_ms"] = (time.perf_counter() - t0) * 1000.0
            else:
                pool_scores[name] = dense[name][pool].astype(np.float64)
        t0 = time.perf_counter()
        valid = np.ones((1, pool.size), dtype=bool)
        normed = {m: normalize_scores(pool_scores[m][None, :], valid, self.hybrid["normalization"])
                  for m in score_models}
        fused = fuse_scores(normed, self.hybrid["weights"])[0]
        mmr = self.hybrid["mmr"]
        vectors = self.release.arrays.get("content") if mmr["enabled"] and self._active("content") else None
        pos = mmr_rerank(fused, pool, vectors, k, float(mmr["lambda"]), int(mmr["candidate_depth"]))
        timings["fusion_mmr_ms"] = (time.perf_counter() - t0) * 1000.0
        out_items = pool[pos]
        if out_items.size < k:
            fallback = "pool_smaller_than_k"
            extra = topk_indices(self.popularity, k + out_items.size, np.concatenate([excluded, out_items]))
            out_items = np.concatenate([out_items, extra])[:k]
            return finish(out_items, np.zeros(out_items.size), score_models)
        return finish(out_items, fused[pos], score_models)
'''

ref = types.ModuleType("cinerank_reference_inference")
sys.modules["cinerank_reference_inference"] = ref
exec(compile(REFERENCE_INFERENCE_SRC, "cinerank_reference_inference.py", "exec"), ref.__dict__)
print("reference inference module loaded; sha256:", hashlib.sha256(REFERENCE_INFERENCE_SRC.encode()).hexdigest()[:16])

## 5. Prepare directories and record the environment

Training data (`DATA_ROOT`) and model output (`BUILD_ROOT`) are separate directories. Nothing is overwritten: a pre-existing release or run directory aborts the notebook.

In [ ]:
set_seed(SEED)
for _d in (RELEASE_DIR, RUN_DIR):
    if _d.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {_d}")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RELEASE_DIR.mkdir(parents=True)
RUN_DIR.mkdir(parents=True)
assert not str(DATA_ROOT).startswith(str(BUILD_ROOT)) and not str(BUILD_ROOT).startswith(str(DATA_ROOT)), \
    "training data and model output directories must be separate"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENVIRONMENT = capture_environment()
print(json.dumps({k: ENVIRONMENT[k] for k in ("python", "numpy", "scipy", "torch", "torch_cuda", "onnxruntime",
                                              "onnxruntime_available_providers", "implicit", "cpu_count", "cuda_available")}, indent=1))
print("attention device:", DEVICE)
print("release dir :", RELEASE_DIR)
print("run dir     :", RUN_DIR)

## 6. Download MovieLens and extract safely

The archive SHA-256 is recorded in the run directory. `ml-latest` is a moving target; the recorded checksum is what makes a run attributable to a specific dataset snapshot.

In [ ]:
def file_sha256(path, chunk=1 << 20):
    return ref.file_sha256(path, chunk)


def download_file(url, destination, chunk_size=1 << 20, connect_timeout=30, read_timeout=300):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0 and zipfile.is_zipfile(destination):
        print(f"Using existing download: {destination} ({destination.stat().st_size:,} bytes)")
        return destination
    tmp_path = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=(connect_timeout, read_timeout)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with open(tmp_path, "wb") as handle, tqdm(total=total or None, unit="B", unit_scale=True, desc="MovieLens download") as bar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    handle.write(chunk)
                    bar.update(len(chunk))
    tmp_path.replace(destination)
    return destination


try:
    if not (ML_DIR / "ratings.csv").exists() or not (ML_DIR / "movies.csv").exists():
        download_file(MOVIELENS_URL, ML_ZIP)
        if not zipfile.is_zipfile(ML_ZIP):
            raise RuntimeError("Downloaded MovieLens archive is not a valid ZIP file")
        ref.safe_extract_zip(ML_ZIP, DATA_ROOT)
except requests.RequestException as exc:
    raise RuntimeError(f"MovieLens data is unavailable ({exc}). Check network access or place the extracted "
                       f"folder at {ML_DIR}.") from exc

assert (ML_DIR / "ratings.csv").is_file(), "ratings.csv missing after extraction"
assert (ML_DIR / "movies.csv").is_file(), "movies.csv missing after extraction"
DATASET_INFO = {"name": "MovieLens", "url": MOVIELENS_URL, "directory": ML_DIR.name,
                "archive_sha256": file_sha256(ML_ZIP) if ML_ZIP.exists() else None,
                "ratings_csv_sha256": file_sha256(ML_DIR / "ratings.csv"),
                "movies_csv_sha256": file_sha256(ML_DIR / "movies.csv")}
print(json.dumps(DATASET_INFO, indent=1))

## 7. Sample users, build the catalogue index, and create the temporal split

**Catalogue.** The full `movies.csv` is the item index (sorted by MovieLens id). Items are *not* removed because they have few ratings in the sampled cohort; frequency statistics are computed from training data only.

**Split (per user, chronological).** Events are ordered by `(timestamp, movieId)` (timestamp ties are broken by movie id; this is deterministic but arbitrary). For users with at least `MIN_POSITIVES_FOR_EVAL` positives:

* **test target** = latest positive event
* **validation target** = second-latest positive event
* **training events** = every event strictly before the validation target (positives and non-positives)
* at validation time the history is the training events; at test time the history is every event before the test target

Events after the test target and (for the training split) events between the validation and test targets are never used for fitting. Users with too few positives contribute all their events to training and are not evaluated.

Known limitation: the split is temporal *per user*, not by a single global date, so other users' later behaviour can inform training. A global-cutoff evaluation is stricter and is listed as future work.

In [ ]:
def build_split(user_idx, item_idx, rating, ts, pos_threshold, min_pos):
    """Vectorised per-user temporal leave-two-out. Inputs must be sorted by (user, ts, item) with
    compact 0..n_users-1 user indices. Returns a dict of aligned arrays."""
    u = np.asarray(user_idx)
    n = u.size
    is_pos = np.asarray(rating) >= pos_threshold
    starts = np.flatnonzero(np.r_[True, u[1:] != u[:-1]])
    ends = np.r_[starts[1:], n]
    lens = ends - starts
    n_users = starts.size
    assert np.array_equal(u[starts], np.arange(n_users)), "user indices must be compact and sorted"
    order = np.arange(n) - np.repeat(starts, lens)
    cum = np.cumsum(is_pos)
    before = np.r_[0, cum][starts]
    total = np.r_[0, cum][ends] - before
    cum_in_user = cum - np.repeat(before, lens)
    rank_from_end = np.repeat(total, lens) - cum_in_user
    eligible = np.repeat(total >= min_pos, lens)
    is_test = is_pos & (rank_from_end == 0) & eligible
    is_val = is_pos & (rank_from_end == 1) & eligible
    val_order = np.full(n_users, -1, dtype=np.int64)
    test_order = np.full(n_users, -1, dtype=np.int64)
    rows = np.flatnonzero(is_val)
    val_order[np.searchsorted(starts, rows, side="right") - 1] = order[rows]
    rows = np.flatnonzero(is_test)
    test_order[np.searchsorted(starts, rows, side="right") - 1] = order[rows]
    val_rep = np.repeat(val_order, lens)
    train_mask = (val_rep < 0) | (order < val_rep)
    n_pos_train = np.bincount(u[train_mask & is_pos], minlength=n_users)
    return {"starts": starts, "ends": ends, "order": order, "is_pos": is_pos, "train_mask": train_mask,
            "val_order": val_order, "test_order": test_order, "n_pos_train": n_pos_train, "n_users": n_users}


def read_movielens_sample():
    """Sample real users (never synthetic), keep the full catalogue, de-duplicate (user, movie) keeping the latest."""
    ratings_path, movies_path = ML_DIR / "ratings.csv", ML_DIR / "movies.csv"
    counts = np.zeros(1, dtype=np.int64)
    total_rows = 0
    for chunk in pd.read_csv(ratings_path, usecols=["userId"], dtype={"userId": "int32"}, chunksize=RATINGS_CHUNK_SIZE):
        bc = np.bincount(chunk["userId"].to_numpy())
        if bc.size > counts.size:
            counts = np.pad(counts, (0, bc.size - counts.size))
        counts[:bc.size] += bc
        total_rows += len(chunk)
    frame = np.flatnonzero(counts >= MIN_USER_RATINGS)
    print(f"Raw dataset: {total_rows:,} ratings, {int((counts > 0).sum()):,} users; "
          f"{frame.size:,} users have >= {MIN_USER_RATINGS} ratings")
    if TARGET_USERS is None or TARGET_USERS >= frame.size:
        selected = frame
        sampling = "all users in the sampling frame"
    else:
        selected = np.sort(np.random.default_rng(SEED).choice(frame, size=TARGET_USERS, replace=False))
        sampling = f"uniform random sample without replacement, seed={SEED}"
    selected_mask = np.zeros(counts.size, dtype=bool)
    selected_mask[selected] = True
    cols = ["userId", "movieId", "rating", "timestamp"]
    dtypes = {"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"}
    pieces = [c[selected_mask[c["userId"].to_numpy()]] for c in
              pd.read_csv(ratings_path, usecols=cols, dtype=dtypes, chunksize=RATINGS_CHUNK_SIZE)]
    ratings = pd.concat([p for p in pieces if len(p)], ignore_index=True)
    del pieces
    movies = pd.read_csv(movies_path, dtype={"movieId": "int64"}).sort_values("movieId", kind="stable").reset_index(drop=True)
    if movies["movieId"].duplicated().any():
        raise RuntimeError("movies.csv contains duplicate movieId values")
    known = ratings["movieId"].isin(movies["movieId"])
    n_unknown = int((~known).sum())
    ratings = ratings[known]
    ratings = ratings.sort_values("timestamp", kind="stable").drop_duplicates(["userId", "movieId"], keep="last")
    info = {"sampling": sampling, "target_users": TARGET_USERS, "seed": SEED, "min_user_ratings": MIN_USER_RATINGS,
            "raw_rows": int(total_rows), "ratings_with_unknown_movie_dropped": n_unknown,
            "sampling_frame_users": int(frame.size)}
    return ratings.reset_index(drop=True), movies, info

In [ ]:
t0 = time.perf_counter()
ratings_raw, movies_df, SAMPLING_INFO = read_movielens_sample()

source_movie_ids = movies_df["movieId"].to_numpy(dtype=np.int64)            # row i of every item-indexed artifact
n_items = int(source_movie_ids.size)
_users_sorted = np.sort(ratings_raw["userId"].unique())
_u = np.searchsorted(_users_sorted, ratings_raw["userId"].to_numpy()).astype(np.int32)
_i = np.searchsorted(source_movie_ids, ratings_raw["movieId"].to_numpy(dtype=np.int64)).astype(np.int32)
assert np.array_equal(source_movie_ids[_i], ratings_raw["movieId"].to_numpy(dtype=np.int64))
_r = ratings_raw["rating"].to_numpy(dtype=np.float32)
_t = ratings_raw["timestamp"].to_numpy(dtype=np.int64)
del ratings_raw
_order = np.lexsort((_i, _t, _u))                                            # (user, timestamp, item)
_u, _i, _r, _t = _u[_order], _i[_order], _r[_order], _t[_order]
del _order

split = build_split(_u, _i, _r, _t, POSITIVE_RATING_THRESHOLD, MIN_POSITIVES_FOR_EVAL)
ev = types.SimpleNamespace(user=_u, item=_i, rating=_r, ts=_t, n_items=n_items, **split)
del _u, _i, _r, _t, split
free_memory()

# ----- feedback-semantics summary -------------------------------------------------------------------
n_events = int(ev.user.size)
n_pos_ev = int((ev.rating >= POSITIVE_RATING_THRESHOLD).sum())
n_neg_ev = int((ev.rating <= NEGATIVE_RATING_THRESHOLD).sum())
n_neu_ev = n_events - n_pos_ev - n_neg_ev
eval_user_ids = np.flatnonzero(ev.test_order >= 0)
train_pos_mask = ev.train_mask & ev.is_pos
item_pos_count = np.bincount(ev.item[train_pos_mask], minlength=n_items).astype(np.int64)
DATA_SUMMARY = {
    "sampling": SAMPLING_INFO, "n_users": int(ev.n_users), "n_items_catalogue": n_items, "n_events": n_events,
    "positive_events": n_pos_ev, "neutral_events": n_neu_ev, "negative_events": n_neg_ev,
    "positive_threshold": POSITIVE_RATING_THRESHOLD, "negative_threshold": NEGATIVE_RATING_THRESHOLD,
    "observed_density": n_events / (ev.n_users * n_items),
    "missing_share": 1.0 - n_events / (ev.n_users * n_items),
    "train_events": int(ev.train_mask.sum()), "train_positive_events": int(train_pos_mask.sum()),
    "items_with_train_positive": int((item_pos_count > 0).sum()),
    "items_without_train_positive": int((item_pos_count == 0).sum()),
    "evaluable_users": int(eval_user_ids.size),
    "users_without_eval_targets": int(ev.n_users - eval_user_ids.size),
    "split_policy": "per-user temporal leave-two-out on positives; train = events before validation target",
    "timestamp_tie_break": "movieId ascending",
    "seconds": round(time.perf_counter() - t0, 1),
}
print(json.dumps(DATA_SUMMARY, indent=1))
print(f"\nMissing observations are NOT dislikes: {DATA_SUMMARY['missing_share']:.4%} of user-item pairs are unobserved.")

In [ ]:
# Sanity checks on the split (no leakage, correct roles).
for _u0 in eval_user_ids[:2000]:
    s, vo, to = ev.starts[_u0], ev.val_order[_u0], ev.test_order[_u0]
    assert 0 < vo < to, "validation target must precede the test target and leave a training history"
    assert ev.is_pos[s + vo] and ev.is_pos[s + to]
    assert not ev.train_mask[s + vo:ev.ends[_u0]].any(), "training events must end before the validation target"
    assert ev.ts[s + vo] >= ev.ts[s + vo - 1] and ev.ts[s + to] >= ev.ts[s + vo], "timestamps must be non-decreasing"
    assert ev.n_pos_train[_u0] >= 1
_cohort_counts = pd.cut(ev.n_pos_train[eval_user_ids], bins=list(COHORT_EDGES) + [np.inf], right=False,
                        labels=COHORT_NAMES).value_counts().reindex(COHORT_NAMES)
print("evaluable users per cohort (by train positives):")
display(_cohort_counts.to_frame("users"))

## 8. Training matrices, explicit-rating diagnostic and evaluation sets

* `train_bin`: binary positive interactions (ItemKNN, BPR).
* `train_conf`: ALS confidence `c = 1 + alpha * relevance`, **positive interactions only**.
* Ratings below the positive threshold never enter these matrices. They are kept as explicit information: they populate exclusion sets (an item the user rated, even low, is not re-recommended), the popularity quality denominator, and an explicit-rating bias baseline that is evaluated on ratings *including* negative ones.

Evaluation sets share users, history, exclusion rules and candidate catalogue across all models. `EVAL_VAL` is for tuning; `EVAL_TEST` is touched exactly once at the end.

In [ ]:
def fit_bias_baseline(user, item, rating, n_users, n_items, damping=10.0):
    """Explicit-rating baseline over ALL ratings (negatives included): mu + item bias + user bias."""
    mu = float(rating.mean())
    resid = rating - mu
    item_sum = np.bincount(item, weights=resid, minlength=n_items)
    item_cnt = np.bincount(item, minlength=n_items)
    b_i = item_sum / (item_cnt + damping)
    resid_u = resid - b_i[item]
    user_sum = np.bincount(user, weights=resid_u, minlength=n_users)
    user_cnt = np.bincount(user, minlength=n_users)
    b_u = user_sum / (user_cnt + damping)
    return mu, b_i, b_u


def cohort_index(n_pos_train):
    return np.searchsorted(COHORT_EDGES, n_pos_train, side="right") - 1


def make_evalset(ev_, split, users):
    """Histories and targets for one split. Validation history = training events; test history = all
    events before the test target (so it includes the validation target and anything in between)."""
    hist_items, hist_ratings, target = [], [], []
    for u in users:
        s = ev_.starts[u]
        o = ev_.val_order[u] if split == "val" else ev_.test_order[u]
        hist_items.append(ev_.item[s:s + o].astype(np.int64))
        hist_ratings.append(ev_.rating[s:s + o])
        target.append(int(ev_.item[s + o]))
    pos_items, pos_rel = [], []
    for it, rt in zip(hist_items, hist_ratings):
        m = rt >= POSITIVE_RATING_THRESHOLD
        pos_items.append(it[m])
        pos_rel.append(ref.rating_to_relevance(rt[m]))
    users = np.asarray(users, dtype=np.int64)
    return types.SimpleNamespace(split=split, users=users, hist_items=hist_items, hist_ratings=hist_ratings,
                                 pos_items=pos_items, pos_rel=pos_rel, target=np.asarray(target, dtype=np.int64),
                                 n_pos_train=ev_.n_pos_train[users], cohort=cohort_index(ev_.n_pos_train[users]))

In [ ]:
_tp = train_pos_mask
pos_user = ev.user[_tp]
pos_item = ev.item[_tp]
pos_rating = ev.rating[_tp]
pos_ts = ev.ts[_tp]
assert (pos_rating >= POSITIVE_RATING_THRESHOLD).all(), "only positive ratings may enter implicit matrices"
pos_rel = ref.rating_to_relevance(pos_rating)
_shape = (ev.n_users, n_items)
train_bin = sp.csr_matrix((np.ones(pos_user.size, dtype=np.float32), (pos_user, pos_item)), shape=_shape)
train_conf = sp.csr_matrix(((1.0 + IMPLICIT_CONFIDENCE_ALPHA * pos_rel).astype(np.float32), (pos_user, pos_item)), shape=_shape)
assert train_bin.nnz == pos_user.size == train_conf.nnz, "duplicate (user,item) positives"
assert train_conf.data.min() > 1.0 and train_bin.data.min() == 1.0
for _m in (train_bin, train_conf):
    _m.sort_indices()
print(f"positive-only matrices: shape={train_bin.shape}, nnz={train_bin.nnz:,} "
      f"(of {int(ev.train_mask.sum()):,} training events; {int(ev.train_mask.sum()) - train_bin.nnz:,} non-positive events excluded)")

# ----- explicit-rating diagnostic: uses ALL training ratings, evaluated on later ratings --------------------
_tm = ev.train_mask
mu_g, b_item, b_user = fit_bias_baseline(ev.user[_tm], ev.item[_tm], ev.rating[_tm], ev.n_users, n_items)
_vo = np.repeat(ev.val_order, ev.ends - ev.starts)
_to = np.repeat(ev.test_order, ev.ends - ev.starts)
_mid = (_vo >= 0) & (ev.order >= _vo) & (ev.order < _to)       # validation target and events before the test target
_pred_bias = np.clip(mu_g + b_item[ev.item[_mid]] + b_user[ev.user[_mid]], 0.5, 5.0)
_y = ev.rating[_mid]
EXPLICIT_DIAGNOSTIC = {
    "rows": int(_mid.sum()),
    "rmse_global_mean": float(np.sqrt(np.mean((_y - mu_g) ** 2))),
    "rmse_user_item_bias": float(np.sqrt(np.mean((_y - _pred_bias) ** 2))),
    "note": "Explicit-rating baseline trained on all training ratings (negatives included) and evaluated on later ratings "
            "from the validation period. It is a diagnostic, not a production model.",
}
del _vo, _to, _mid, _pred_bias, _y
print(json.dumps(EXPLICIT_DIAGNOSTIC, indent=1))

# ----- evaluation users (stratified by cohort, deterministic) -----------------------------------------
_rng = np.random.default_rng(SEED + 1)
_picked = []
_coh_all = cohort_index(ev.n_pos_train[eval_user_ids])
for _c in range(len(COHORT_NAMES)):
    _pool_users = eval_user_ids[_coh_all == _c]
    if _pool_users.size:
        _picked.append(np.sort(_rng.choice(_pool_users, size=min(EVAL_USERS_PER_COHORT, _pool_users.size), replace=False)))
EVAL_USERS = np.sort(np.concatenate(_picked))
EVAL_VAL = make_evalset(ev, "val", EVAL_USERS)
EVAL_TEST = make_evalset(ev, "test", EVAL_USERS)
TUNE_IDX = np.sort(np.random.default_rng(SEED + 2).choice(len(EVAL_USERS), size=min(TUNE_USERS, len(EVAL_USERS)), replace=False))

# ----- evaluation context (item frequency from TRAINING data only) -----------------------------------------
BAND_OF_ITEM = np.searchsorted(ITEM_BAND_EDGES, item_pos_count, side="right")
POP_PROB = (item_pos_count + 1.0) / (item_pos_count.sum() + n_items)
EVALCTX = types.SimpleNamespace(n_items=n_items, pop_prob=POP_PROB, band_of_item=BAND_OF_ITEM,
                                longtail_item=BAND_OF_ITEM <= 1, content=None)
_tgt_band = BAND_OF_ITEM[EVAL_VAL.target]
print(f"evaluation users: {len(EVAL_USERS):,}; validation targets with ZERO train positives: "
      f"{int((_tgt_band == 0).sum()):,} ({(_tgt_band == 0).mean():.1%}); all targets are inside the item index (index = full catalogue)")
display(pd.DataFrame({"validation targets": np.bincount(_tgt_band, minlength=4)}, index=ITEM_BAND_NAMES))
del _picked, _rng, _coh_all, _tp

## 9. Evaluation framework

* **Ranking.** Every model scores the full catalogue (protocol A) or a shared candidate pool (protocol B). History items (all prior events, including low-rated ones) are excluded identically for every model.
* **Ties** are resolved pessimistically (an item with a score equal to the target's counts as ranked above it), so degenerate models are not rewarded.
* **One held-out target per user.** With a single relevant item, `MAP@K` and `MRR@K` are mathematically identical; both are reported because both were requested.
* **Coverage** = distinct items recommended in the top 10 / catalogue size. **Diversity** = mean intra-list `1 - cosine` of content vectors. **Novelty** = mean `-log2` of smoothed training popularity. **Long-tail exposure** = share of recommended items with at most 10 training positives.
* **Candidate pool (protocol B).** Union of the top `CANDIDATE_POOL_PER_SOURCE` items from each generator (popularity, content, ItemKNN, ALS, plus BPR only if production-enabled). The attention model and the hybrid are re-rankers, so every model is also reported on the same pool. Pool recall is the ceiling for any re-ranker.

In [ ]:
def accuracy_metrics(ranks, ks=KS):
    r = np.asarray(ranks, dtype=np.float64)
    out = {}
    with np.errstate(divide="ignore", invalid="ignore"):
        for k in ks:
            hit = r <= k
            gain = np.where(hit, 1.0 / np.log2(r + 1.0), 0.0)
            recip = np.where(hit, 1.0 / r, 0.0)
            out[f"recall@{k}"] = float(hit.mean())
            out[f"precision@{k}"] = float((hit / k).mean())
            out[f"ndcg@{k}"] = float(gain.mean())
            out[f"map@{k}"] = float(recip.mean())      # one relevant item: AP@K == 1/rank == RR@K
            out[f"mrr@{k}"] = float(recip.mean())
    return out


def enrich_result(result):
    """Attach per-user beyond-accuracy values (computed once per result)."""
    topk = result["topk"][:, :10]
    U = topk.shape[0]
    ild = np.full(U, np.nan)
    nov = np.full(U, np.nan)
    ltail = np.full(U, np.nan)
    V = EVALCTX.content
    for u in range(U):
        items = topk[u][topk[u] >= 0]
        if items.size == 0:
            continue
        nov[u] = float(np.mean(-np.log2(EVALCTX.pop_prob[items])))
        ltail[u] = float(EVALCTX.longtail_item[items].mean())
        if V is not None and items.size >= 2:
            m = V[items]
            sim = np.asarray((m @ m.T).todense())
            n = items.size
            off = sim.sum() - np.trace(sim)
            ild[u] = float(1.0 - off / (n * (n - 1)))
    result["ild"], result["novelty"], result["longtail"] = ild, nov, ltail
    return result


def summarize_result(result, mask=None):
    U = result["ranks"].shape[0]
    m = np.ones(U, dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    if "ild" not in result:
        enrich_result(result)
    out = accuracy_metrics(result["ranks"][m])
    lists = result["topk"][m][:, :10]
    shown = lists[lists >= 0]
    out["coverage@10"] = float(np.unique(shown).size / EVALCTX.n_items) if shown.size else 0.0
    out["diversity@10"] = float(np.nanmean(result["ild"][m])) if np.isfinite(result["ild"][m]).any() else float("nan")
    out["novelty@10"] = float(np.nanmean(result["novelty"][m])) if np.isfinite(result["novelty"][m]).any() else float("nan")
    out["long_tail_exposure@10"] = float(np.nanmean(result["longtail"][m])) if np.isfinite(result["longtail"][m]).any() else float("nan")
    out["n_users"] = int(m.sum())
    return out


def full_report(result, es):
    """Overall metrics, per history cohort, and per target-popularity band."""
    rep = {"overall": summarize_result(result), "cohorts": {}, "target_bands": {}}
    for c, name in enumerate(COHORT_NAMES):
        mask = es.cohort == c
        if mask.any():
            rep["cohorts"][name] = summarize_result(result, mask)
    band = BAND_OF_ITEM[es.target]
    for b, name in enumerate(ITEM_BAND_NAMES):
        mask = band == b
        if mask.any():
            s = accuracy_metrics(result["ranks"][mask])
            rep["target_bands"][name] = {"n_users": int(mask.sum()), "recall@10": s["recall@10"], "ndcg@10": s["ndcg@10"],
                                         "recall@20": s["recall@20"], "mrr@20": s["mrr@20"]}
    return rep


def ranks_to_result(ranks, topk):
    return {"ranks": np.asarray(ranks, dtype=np.float64), "topk": np.asarray(topk, dtype=np.int32)}


def rank_dense(S, es, chunk, want_topk=False):
    """Mask each user's history in place, then rank the target (pessimistic ties)."""
    B = S.shape[0]
    np.nan_to_num(S, copy=False, nan=-np.inf, posinf=np.finfo(np.float32).max)
    for r, i in enumerate(chunk):
        S[r, es.hist_items[i]] = -np.inf
    ts = S[np.arange(B), es.target[chunk]]
    ranks = (S >= ts[:, None]).sum(axis=1).astype(np.float64)
    if not want_topk:
        return ranks, None
    top = np.full((B, KMAX), -1, dtype=np.int32)
    for r in range(B):
        t = ref.topk_indices(S[r], KMAX)
        top[r, :t.size] = t
    return ranks, top


def dense_scores(M, name, es, chunk):
    """Dense [B x n_items] scores for one model. M carries fitted state (see MODELS)."""
    B, n = len(chunk), M.n_items
    if name == "random":
        return np.random.default_rng(SEED + 7919 * int(chunk[0])).random((B, n), dtype=np.float32)
    if name == "popularity":
        return np.repeat(M.pop[None, :], B, axis=0)
    if name in ("content", "itemknn"):
        H = ref.history_matrix([es.pos_items[i] for i in chunk], [None] * B, n)
        return ref.content_scores(M.content, H) if name == "content" else ref.itemknn_scores(M.knn, H)
    if name == "als":
        X = np.stack([ref.als_fold_in(M.als_Y, M.als_gram, es.pos_items[i], es.pos_rel[i], M.als_alpha, M.als_reg) for i in chunk])
        return (X @ M.als_Y.T).astype(np.float32)
    if name == "bpr":
        fi = M.bpr_fold
        X = np.stack([ref.bpr_fold_in(M.bpr_Y, es.pos_items[i], es.hist_items[i], fi["iterations"], fi["learning_rate"],
                                      fi["regularization"], fi["negatives_per_positive"],
                                      ref.bpr_seed(fi["seed"], es.pos_items[i]), fi["max_positives"]) for i in chunk])
        return ref.bpr_scores(M.bpr_Y, X)
    raise KeyError(name)


def quick_eval(M, name, es, idx, k=10):
    """Fast full-catalogue evaluation used for hyperparameter selection on validation users."""
    ranks = []
    for c0 in range(0, len(idx), EVAL_CHUNK):
        chunk = np.asarray(idx[c0:c0 + EVAL_CHUNK])
        r, _ = rank_dense(dense_scores(M, name, es, chunk), es, chunk)
        ranks.append(r)
    m = accuracy_metrics(np.concatenate(ranks), ks=(k,))
    return {f"ndcg@{k}": m[f"ndcg@{k}"], f"recall@{k}": m[f"recall@{k}"], f"mrr@{k}": m[f"mrr@{k}"]}


def run_split_evaluation(M, es, model_names, generators, per_source, desc="evaluating"):
    """One pass over users: full-catalogue ranks/top-K for every model (protocol A) and the shared
    candidate pool with every model's pool scores (protocol B)."""
    U = len(es.users)
    full = {m: {"ranks": np.full(U, np.inf), "topk": np.full((U, KMAX), -1, dtype=np.int32)} for m in model_names}
    pools, pool_scores = [], {m: [] for m in model_names}
    for c0 in tqdm(range(0, U, EVAL_CHUNK), desc=desc):
        chunk = np.arange(c0, min(U, c0 + EVAL_CHUNK))
        dense = {}
        for m in model_names:
            S = dense_scores(M, m, es, chunk)
            r, top = rank_dense(S, es, chunk, want_topk=True)
            full[m]["ranks"][chunk] = r
            full[m]["topk"][chunk] = top
            dense[m] = S                                       # history already masked to -inf
        for r_i in range(len(chunk)):
            members = [ref.topk_indices(dense[g][r_i], per_source) for g in generators]
            pool = np.unique(np.concatenate(members))
            pools.append(pool)
            for m in model_names:
                pool_scores[m].append(dense[m][r_i][pool].copy())
        del dense
    P = max(len(p) for p in pools)
    items = np.zeros((U, P), dtype=np.int32)
    valid = np.zeros((U, P), dtype=bool)
    scores = {m: np.zeros((U, P), dtype=np.float32) for m in model_names}
    target_col = np.full(U, -1, dtype=np.int64)
    for u, p in enumerate(pools):
        items[u, :p.size] = p
        valid[u, :p.size] = True
        for m in model_names:
            scores[m][u, :p.size] = pool_scores[m][u]
        j = np.searchsorted(p, es.target[u])
        if j < p.size and p[j] == es.target[u]:
            target_col[u] = j
    pd_ = types.SimpleNamespace(items=items, valid=valid, scores=scores, target_col=target_col,
                                sizes=np.asarray([p.size for p in pools]), generators=tuple(generators))
    return full, pd_


def pool_rank(scores, pool):
    """Rank every user's target inside the shared candidate pool (items absent from the pool are misses)."""
    s = np.where(pool.valid, scores, -np.inf).astype(np.float64)
    U = s.shape[0]
    tc = pool.target_col
    ts = s[np.arange(U), np.maximum(tc, 0)]
    ranks = (s >= ts[:, None]).sum(axis=1).astype(np.float64)
    ranks[tc < 0] = np.inf
    order = np.argsort(-s, axis=1, kind="stable")[:, :KMAX]
    top = np.take_along_axis(pool.items, order, axis=1).astype(np.int32)
    ok = np.take_along_axis(pool.valid, order, axis=1) & np.isfinite(np.take_along_axis(s, order, axis=1))
    top[~ok] = -1
    return ranks_to_result(ranks, top)


def results_frame(results, metrics=("recall@10", "ndcg@10", "coverage@10", "diversity@10", "novelty@10", "long_tail_exposure@10")):
    return pd.DataFrame({m: {k: v["overall"][k] for k in metrics} for m, v in results.items()}).T

## 10. Popularity

Signals from **training positives**, anchored to a fixed cutoff `t_cut` = the latest training timestamp (never the wall-clock date, so the same data produces the same artifact):

* `D_i = sum_{positive events} relevance * exp(-lambda * (t_cut - t))`: time-decayed positive weight.
* `q_i = (P_i + m * q0) / (A_i + m)`: Bayesian-smoothed share of exposures that were positive, where `A_i` counts *all* decayed training ratings of the item (low ratings are exposures that were not positive), `q0` is the global share and `m = POP_PRIOR_STRENGTH`.
* `score_i = D_i * q_i`, scaled by its maximum to `[0, 1]`.

Items with no positive observation score exactly 0 (the backend treats `score == 0` as "no collaborative signal"). Ties break by ascending item index.

In [ ]:
def fit_popularity(pos_item_, pos_rel_, pos_ts_, all_item_, all_ts_, n_items_, t_cut, half_life_days, prior_m):
    lam = math.log(2.0) / (half_life_days * 86400.0)
    d_pos = np.exp(-lam * np.maximum(t_cut - pos_ts_, 0)).astype(np.float64)
    d_all = np.exp(-lam * np.maximum(t_cut - all_ts_, 0)).astype(np.float64)
    D = np.bincount(pos_item_, weights=pos_rel_ * d_pos, minlength=n_items_)
    P = np.bincount(pos_item_, weights=d_pos, minlength=n_items_)
    A = np.bincount(all_item_, weights=d_all, minlength=n_items_)
    q0 = float(P.sum() / A.sum())
    q = (P + prior_m * q0) / (A + prior_m)
    score = D * q
    scale = float(score.max())
    if not scale > 0:
        raise RuntimeError("popularity has no positive signal")
    return (score / scale).astype(np.float32), {"lambda_per_second": lam, "q0": q0, "scale": scale}


def zero_pop_check(scores, counts):
    assert np.array_equal(scores > 0, counts > 0), "popularity must be > 0 exactly where positives exist"

In [ ]:
MODELS = types.SimpleNamespace(n_items=n_items, content=None, knn=None, als_Y=None, als_gram=None,
                               als_alpha=IMPLICIT_CONFIDENCE_ALPHA, als_reg=None, bpr_Y=None, bpr_fold=None, pop=None)
TRAINING_RECORDS = {}          # everything diagnostic that goes to metrics.json
SELECTED = {}                  # hyperparameters chosen on validation

with model_stage("popularity", critical=True):
    T_CUT = int(ev.ts[ev.train_mask].max())
    popularity_scores, _pop_info = fit_popularity(pos_item, pos_rel, pos_ts, ev.item[ev.train_mask], ev.ts[ev.train_mask],
                                                  n_items, T_CUT, POP_HALF_LIFE_DAYS, POP_PRIOR_STRENGTH)
    zero_pop_check(popularity_scores, item_pos_count)
    assert np.isfinite(popularity_scores).all() and popularity_scores.dtype == np.float32
    MODELS.pop = popularity_scores
    POP_CONFIG = {"schema_version": 1, "model": "popularity", "t_cut_unix": T_CUT,
                  "t_cut_utc": datetime.fromtimestamp(T_CUT, timezone.utc).isoformat(),
                  "half_life_days": POP_HALF_LIFE_DAYS, "prior_strength": POP_PRIOR_STRENGTH, **_pop_info,
                  "formula": "score_i = D_i * q_i / max; D_i = sum_pos relevance*exp(-lambda*(t_cut-t)); "
                             "q_i = (P_i + m*q0)/(A_i + m)",
                  "positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
                  "unobserved_item_score": 0.0, "tie_break": "descending score, then ascending item index",
                  "items_with_signal": int((popularity_scores > 0).sum())}
    print(f"popularity: {POP_CONFIG['items_with_signal']:,} of {n_items:,} items have a positive signal; cutoff {POP_CONFIG['t_cut_utc']}")

## 11. Content-based TF-IDF

Features per catalogue item (MovieLens exposes only these): title words (`t:`), title bigrams (`tb:`), genres (`g:`) and release decade (`d:`). Tokenization is NFKD ASCII folding, lower-casing, `[a-z0-9]+` words, a fixed stop-word list, with the trailing `(YYYY)` removed from the title and parsed as the year. Weights: `block_weight * idf`, binary term frequency, L2-normalised. Vocabulary and IDF are fitted on **catalogue text only** (no interaction data). The identical function builds vectors for training items and for items that appear in the backend later; the new-item contract is tested in the validation section.

Richer metadata (overview, keywords, cast, director) would improve this model but is not part of MovieLens; adding it needs a licensed source and an identifier mapping.

In [ ]:
with model_stage("content"):
    _tokens = [ref.content_tokens(t, g) for t, g in zip(movies_df["title"].fillna(""), movies_df["genres"].fillna(""))]
    content_vocab, content_idf = ref.fit_content_vocabulary(_tokens, CONTENT_MIN_DF, CONTENT_MAX_FEATURES)
    content_vectors = ref.content_vectors(_tokens, content_vocab, content_idf, CONTENT_BLOCK_WEIGHTS)
    del _tokens
    _norms = np.sqrt(np.asarray(content_vectors.multiply(content_vectors).sum(axis=1)).ravel())
    assert content_vectors.shape == (n_items, len(content_vocab))
    assert np.all(np.isclose(_norms, 1.0, atol=1e-4) | (_norms == 0)), "content rows must be unit norm or empty"
    MODELS.content = content_vectors
    EVALCTX.content = content_vectors
    print(f"content vectors: {content_vectors.shape}, nnz={content_vectors.nnz:,}, "
          f"empty rows={int((_norms == 0).sum())}")
    TRAINING_RECORDS["content"] = {"vocabulary_size": len(content_vocab), "nnz": int(content_vectors.nnz),
                                   "empty_rows": int((_norms == 0).sum())}
    _q = quick_eval(MODELS, "content", EVAL_VAL, TUNE_IDX)
    print("content validation (tune subset, full catalogue):", {k: round(v, 4) for k, v in _q.items()})
    _q_pop = quick_eval(MODELS, "popularity", EVAL_VAL, TUNE_IDX)
    print("popularity validation (same users):             ", {k: round(v, 4) for k, v in _q_pop.items()})
    POP_VAL_NDCG10 = _q_pop["ndcg@10"]

## 12. ItemKNN (sparse, shrunk, top-K)

* Built from **positive** interactions only (`train_bin`). Rows are item vectors over users; similarity is cosine over (optionally user-down-weighted) vectors.
* `user_idf` weighting multiplies each user's contribution by `1/log2(2 + n_positives)` so heavy users do not dominate; `binary` is plain cosine. Both are compared on validation.
* **Shrinkage:** `sim' = sim * n_ij / (n_ij + s)` where `n_ij` is the co-occurrence count, damping similarities supported by few users.
* Only the top `K` neighbours per item are retained; no dense item x item matrix is ever created. The intermediate product is computed in row blocks and peak RSS is recorded, because sparse inputs do not guarantee small products.
* K in {20, 50, 100} is chosen on validation NDCG@10. Candidate recall (target inside the model's own top 100) is reported later, separately from ranking quality.

In [ ]:
def build_itemknn(user_item_bin, weighting, k_max, shrinkage, row_block, min_support):
    X = user_item_bin.tocsr()
    n_users_, n_items_ = X.shape
    degree = np.asarray(X.sum(axis=1)).ravel()
    if weighting == "binary":
        uw = np.ones(n_users_, dtype=np.float32)
    elif weighting == "user_idf":
        uw = (1.0 / np.log2(2.0 + degree)).astype(np.float32)
    else:
        raise ValueError(weighting)
    item_user_w = (sp.diags(uw) @ X).T.tocsr()
    norms = np.sqrt(np.asarray(item_user_w.multiply(item_user_w).sum(axis=1)).ravel())
    inv = np.divide(1.0, norms, out=np.zeros_like(norms), where=norms > 0).astype(np.float32)
    item_user_w = (sp.diags(inv) @ item_user_w).tocsr().astype(np.float32)
    item_user_b = X.T.tocsr()
    support = np.diff(item_user_b.indptr)
    rows_out, cols_out, vals_out = [], [], []
    max_block_nnz = 0
    for b0 in range(0, n_items_, row_block):
        b1 = min(n_items_, b0 + row_block)
        active = np.flatnonzero(support[b0:b1] >= min_support)
        if active.size == 0:
            continue
        S = (item_user_w[b0:b1] @ item_user_w.T).tocsr()
        N = (item_user_b[b0:b1] @ item_user_b.T).tocsr()
        max_block_nnz = max(max_block_nnz, S.nnz)
        N.data = N.data / (N.data + shrinkage)
        S = S.multiply(N).tocsr()
        for r in active:
            a, b = S.indptr[r], S.indptr[r + 1]
            cols, vals = S.indices[a:b], S.data[a:b]
            keep = (cols != b0 + r) & (support[cols] >= min_support)
            cols, vals = cols[keep], vals[keep]
            if vals.size == 0:
                continue
            sel = np.lexsort((cols, -vals))[:k_max]
            rows_out.append(np.full(sel.size, b0 + r, dtype=np.int32))
            cols_out.append(cols[sel].astype(np.int32))
            vals_out.append(vals[sel].astype(np.float32))
        del S, N
    if not rows_out:
        raise RuntimeError("ItemKNN produced no neighbours")
    out = sp.csr_matrix((np.concatenate(vals_out), (np.concatenate(rows_out), np.concatenate(cols_out))),
                        shape=(n_items_, n_items_), dtype=np.float32)
    out.sort_indices()
    return out, max_block_nnz


def truncate_topk_rows(S, k):
    """Keep the k largest entries of every row (ties: lower column first)."""
    S = S.tocsr()
    if int(np.diff(S.indptr).max()) <= k:
        return S.copy()
    rows, cols, vals = [], [], []
    for r in range(S.shape[0]):
        a, b = S.indptr[r], S.indptr[r + 1]
        if a == b:
            continue
        c, v = S.indices[a:b], S.data[a:b]
        sel = np.lexsort((c, -v))[:k]
        rows.append(np.full(sel.size, r, dtype=np.int32)); cols.append(c[sel]); vals.append(v[sel])
    out = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=S.shape, dtype=np.float32)
    out.sort_indices()
    return out

In [ ]:
with model_stage("itemknn"):
    knn_grid, best = [], None
    for weighting in KNN_WEIGHTINGS:
        for shrink in KNN_SHRINKAGE_GRID:
            t0 = time.perf_counter()
            S_full, blk = build_itemknn(train_bin, weighting, KNN_K_MAX, shrink, KNN_ROW_BLOCK, KNN_MIN_ITEM_SUPPORT)
            build_s = time.perf_counter() - t0
            for k in KNN_K_GRID:
                S_k = truncate_topk_rows(S_full, k)
                q = quick_eval(types.SimpleNamespace(**{**vars(MODELS), "knn": S_k}), "itemknn", EVAL_VAL, TUNE_IDX)
                row = {"weighting": weighting, "shrinkage": shrink, "k": k, "nnz": int(S_k.nnz), "build_seconds": round(build_s, 1),
                       "max_block_nnz": int(blk), "peak_rss_mb": round(peak_rss_mb()), **q}
                knn_grid.append(row)
                print({k_: (round(v, 4) if isinstance(v, float) else v) for k_, v in row.items()})
                if best is None or q["ndcg@10"] > best[0]["ndcg@10"]:
                    best = (row, S_k)
            del S_full
            free_memory()
    SELECTED["itemknn"] = best[0]
    itemknn_topk = best[1]
    assert np.isfinite(itemknn_topk.data).all() and itemknn_topk.shape == (n_items, n_items)
    assert not (itemknn_topk.indices == np.repeat(np.arange(n_items), np.diff(itemknn_topk.indptr))).any(), "self-similarity present"
    MODELS.knn = itemknn_topk
    TRAINING_RECORDS["itemknn_grid"] = knn_grid
    print("selected ItemKNN:", {k: v for k, v in best[0].items() if k in ("weighting", "shrinkage", "k", "ndcg@10")})
    display(pd.DataFrame(knn_grid).sort_values("ndcg@10", ascending=False).head(8))

## 13. Implicit ALS with new-user fold-in

* Trained on `train_conf`: **positives only**, `c = 1 + alpha * relevance`. Items without positives keep (near-)zero factors.
* Factors and regularisation are selected on validation NDCG@10, where **every evaluation user is represented by fold-in**, not by a stored user factor:

  `x_u = (Y^T Y + sum_i (c_ui - 1) y_i y_i^T + lambda I)^-1 sum_i c_ui y_i`

* Only item factors are exported. MovieLens user factors are never exported or used for production users.
* BLAS threads are limited to 1 during `implicit` ALS to avoid oversubscription with OpenMP; per-fit wall time is recorded.
* The fold-in is compared against the library's own `recalculate_user` (an exact Cholesky solve of the same normal equations, so it must agree to numerical precision; the notebook asserts median cosine >= 0.99) and against the stored training-user factors (informational: the stored factors lag one half-step behind the final item factors, so exact equality is not expected).

Only part of the hyperparameter space is searched (a small grid); this is "tuned where practical", not exhaustively optimised.

In [ ]:
def _implicit_class(kind):
    """Concrete CPU implementation. In implicit >= 0.5 `implicit.als.AlternatingLeastSquares` is a factory
    function (its signature is *args, **kwargs), so the class itself is resolved to guarantee the CPU path."""
    try:
        if kind == "als":
            from implicit.cpu.als import AlternatingLeastSquares as cls
        else:
            from implicit.cpu.bpr import BayesianPersonalizedRanking as cls
    except ImportError:                       # releases that expose the class directly
        if kind == "als":
            from implicit.als import AlternatingLeastSquares as cls
        else:
            from implicit.bpr import BayesianPersonalizedRanking as cls
    return cls


def _kwargs_for(cls, required=(), **kw):
    """Pass only arguments the class accepts, but never silently fall back to library defaults for
    arguments that define the experiment (`required`)."""
    import inspect
    params = inspect.signature(cls).parameters
    accepted = {k: v for k, v in kw.items() if k in params}
    dropped = [k for k in kw if k not in params]
    missing = [k for k in required if k in dropped]
    if missing:
        raise RuntimeError(f"{cls.__name__} does not accept required arguments {missing}; installed implicit is incompatible")
    return accepted, dropped


def _to_numpy(x):
    return x.to_numpy() if hasattr(x, "to_numpy") else np.asarray(x)


def fit_als(conf_matrix, factors, reg, iterations, seed):
    cls = _implicit_class("als")
    kw, dropped = _kwargs_for(cls, required=("factors", "regularization", "iterations", "random_state"),
                              factors=factors, regularization=reg, iterations=iterations, random_state=seed,
                              use_gpu=False, calculate_training_loss=False, alpha=1.0)   # matrix already holds confidences
    model = cls(**kw)
    with threadpool_limits(limits=1, user_api="blas"):
        model.fit(conf_matrix, show_progress=False)
    return model


def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na > 0 and nb > 0 else float("nan")

In [ ]:
with model_stage("als"):
    als_grid, best_als = [], None
    for factors in ALS_FACTORS_GRID:
        for reg in ALS_REG_GRID:
            for iters in ALS_ITERATIONS_GRID:
                t0 = time.perf_counter()
                model = fit_als(train_conf, factors, reg, iters, SEED)
                fit_s = time.perf_counter() - t0
                Y = _to_numpy(model.item_factors).astype(np.float32)
                Mtmp = types.SimpleNamespace(**{**vars(MODELS), "als_Y": Y, "als_gram": ref.als_gram(Y), "als_reg": reg})
                q = quick_eval(Mtmp, "als", EVAL_VAL, TUNE_IDX)
                row = {"factors": factors, "regularization": reg, "iterations": iters, "fit_seconds": round(fit_s, 1), **q}
                als_grid.append(row)
                print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
                if best_als is None or q["ndcg@10"] > best_als[0]["ndcg@10"]:
                    best_als = (row, model, Y)
                else:
                    del model
                free_memory()
    row, als_model, als_Y = best_als
    SELECTED["als"] = row
    als_reg = row["regularization"]
    als_gram = ref.als_gram(als_Y)
    assert np.isfinite(als_Y).all() and als_Y.dtype == np.float32 and als_Y.shape == (n_items, row["factors"])
    MODELS.als_Y, MODELS.als_gram, MODELS.als_reg = als_Y, als_gram, als_reg
    TRAINING_RECORDS["als_grid"] = als_grid
    print("selected ALS:", row)

    # ----- fold-in validation against the training library --------------------------------------------------
    rng = np.random.default_rng(SEED + 3)
    has_pos = np.flatnonzero(np.diff(train_conf.indptr) > 0)
    sample = rng.choice(has_pos, size=min(60, has_pos.size), replace=False)
    cos_trained, cos_recalc, overlap_trained, recalc_error = [], [], [], None
    user_factors = _to_numpy(als_model.user_factors)
    for u in sample:
        a, b = train_conf.indptr[u], train_conf.indptr[u + 1]
        idx = train_conf.indices[a:b]
        rel = (train_conf.data[a:b] - 1.0) / IMPLICIT_CONFIDENCE_ALPHA
        x = ref.als_fold_in(als_Y, als_gram, idx, rel, IMPLICIT_CONFIDENCE_ALPHA, als_reg)
        cos_trained.append(cosine(x, user_factors[u]))
        s1, s2 = als_Y @ x, als_Y @ user_factors[u]
        s1[idx] = s2[idx] = -np.inf
        overlap_trained.append(len(set(ref.topk_indices(s1, 10)) & set(ref.topk_indices(s2, 10))) / 10.0)
        try:
            xr = _to_numpy(als_model.recalculate_user(int(u), train_conf[int(u)])).ravel()   # one-row matrix, exact Cholesky solve
            cos_recalc.append(cosine(x, xr))
        except Exception as exc:  # noqa: BLE001
            recalc_error = f"{type(exc).__name__}: {exc}"
    ALS_FOLDIN_CHECK = {"users": int(sample.size), "median_cosine_vs_trained_user_factors": float(np.nanmedian(cos_trained)),
                        "median_top10_overlap_vs_trained_user_factors": float(np.median(overlap_trained)),
                        "median_cosine_vs_library_recalculate_user": float(np.nanmedian(cos_recalc)) if cos_recalc else None,
                        "library_recalculate_user_error": recalc_error}
    print(json.dumps(ALS_FOLDIN_CHECK, indent=1))
    if cos_recalc:
        assert ALS_FOLDIN_CHECK["median_cosine_vs_library_recalculate_user"] >= 0.99, \
            "fold-in disagrees with the library's own single-user solve"
    TRAINING_RECORDS["als_foldin_check"] = ALS_FOLDIN_CHECK
    model = None
    del als_model, user_factors

## 14. BPR with pairwise new-user fold-in (production-gated)

* Trained with `implicit` on `train_bin` (positives only). `implicit` verifies sampled negatives are not the user's positives; sampled items are *unobserved*, not known dislikes. Single-threaded for reproducibility (multi-threaded BPR updates are asynchronous).
* Item factors have `factors + 1` columns; the last column is the item bias.
* **Production representation:** the user vector is obtained by optimising the BPR pairwise loss with item factors frozen (Adam, deterministic seed derived from the history, negatives sampled uniformly and never from the user's history). Averaging item factors is *not* used.
* **Gate:** BPR is exported either way, but `production_enabled` is true only if (a) fold-in validation NDCG@10 reaches `popularity * BPR_MIN_RELATIVE_NDCG_VS_POPULARITY` and (b) the fold-in loss decreases. Otherwise BPR stays disabled and is excluded from the hybrid.

In [ ]:
def fit_bpr(binary_matrix, factors, lr, iterations, reg, seed):
    cls = _implicit_class("bpr")
    kw, dropped = _kwargs_for(cls, required=("factors", "learning_rate", "regularization", "iterations", "random_state"),
                              factors=factors, learning_rate=lr, regularization=reg, iterations=iterations,
                              random_state=seed, use_gpu=False, verify_negative_samples=True, num_threads=1)
    model = cls(**kw)
    model.fit(binary_matrix, show_progress=False)
    return model, {"unsupported_arguments_dropped": dropped, "verify_negative_samples": "verify_negative_samples" in kw}

In [ ]:
BPR_PRODUCTION_ENABLED = False
with model_stage("bpr"):
    bpr_grid, best_bpr = [], None
    for lr in BPR_LR_GRID:
        t0 = time.perf_counter()
        model, bpr_fit_info = fit_bpr(train_bin, BPR_FACTORS, lr, BPR_ITERATIONS, BPR_REG, SEED)
        fit_s = time.perf_counter() - t0
        Y = _to_numpy(model.item_factors).astype(np.float32)
        assert Y.shape == (n_items, BPR_FACTORS + 1) and np.isfinite(Y).all()
        for fold in BPR_FOLDIN_GRID:
            fi = {**BPR_FOLDIN_FIXED, **fold}
            q = quick_eval(types.SimpleNamespace(**{**vars(MODELS), "bpr_Y": Y, "bpr_fold": fi}), "bpr", EVAL_VAL, TUNE_IDX[:500])
            row = {"train_learning_rate": lr, "foldin_learning_rate": fold["learning_rate"], "fit_seconds": round(fit_s, 1), **q}
            bpr_grid.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
            if best_bpr is None or q["ndcg@10"] > best_bpr[0]["ndcg@10"]:
                best_bpr = (row, model, Y, fi)
        if best_bpr[1] is not model:
            del model
        free_memory()
    row, bpr_model, bpr_Y, bpr_fold = best_bpr
    SELECTED["bpr"] = row
    MODELS.bpr_Y, MODELS.bpr_fold = bpr_Y, bpr_fold

    # ----- fold-in vs trained user vectors, convergence, gate ---------------------------------------------------
    rng = np.random.default_rng(SEED + 4)
    has_pos = np.flatnonzero(np.diff(train_bin.indptr) > 0)
    sample = rng.choice(has_pos, size=min(BPR_USERS_FOR_TRAINED_COMPARISON, has_pos.size), replace=False)
    trained_uf = _to_numpy(bpr_model.user_factors)
    overlaps = []
    for u in sample:
        idx = train_bin.indices[train_bin.indptr[u]:train_bin.indptr[u + 1]]
        x = ref.bpr_fold_in(bpr_Y, idx, idx, bpr_fold["iterations"], bpr_fold["learning_rate"], bpr_fold["regularization"],
                            bpr_fold["negatives_per_positive"], ref.bpr_seed(bpr_fold["seed"], idx), bpr_fold["max_positives"])
        s1 = ref.bpr_scores(bpr_Y, x)[0]
        s2 = bpr_Y @ trained_uf[u]
        s1[idx] = s2[idx] = -np.inf
        overlaps.append(len(set(ref.topk_indices(s1, 10)) & set(ref.topk_indices(s2, 10))) / 10.0)
    starts_, ends_ = [], []
    for i in TUNE_IDX[:100]:
        pi = EVAL_VAL.pos_items[i]
        _, d = ref.bpr_fold_in(bpr_Y, pi, EVAL_VAL.hist_items[i], bpr_fold["iterations"], bpr_fold["learning_rate"],
                               bpr_fold["regularization"], bpr_fold["negatives_per_positive"],
                               ref.bpr_seed(bpr_fold["seed"], pi), bpr_fold["max_positives"], return_diagnostics=True)
        if d["loss_start"] is not None:
            starts_.append(d["loss_start"]); ends_.append(d["loss_end"])
    converged = bool(np.median(ends_) < np.median(starts_))
    pop_q = quick_eval(MODELS, "popularity", EVAL_VAL, TUNE_IDX[:500])["ndcg@10"]
    beats_pop = row["ndcg@10"] >= pop_q * BPR_MIN_RELATIVE_NDCG_VS_POPULARITY
    BPR_PRODUCTION_ENABLED = bool(beats_pop and converged)
    BPR_GATE = {"foldin_val_ndcg@10": row["ndcg@10"], "popularity_val_ndcg@10_same_users": pop_q,
                "required_ratio": BPR_MIN_RELATIVE_NDCG_VS_POPULARITY, "foldin_loss_median_start": float(np.median(starts_)),
                "foldin_loss_median_end": float(np.median(ends_)), "converged": converged,
                "median_top10_overlap_vs_trained_user_vectors": float(np.median(overlaps)),
                "production_enabled": BPR_PRODUCTION_ENABLED, **bpr_fit_info}
    TRAINING_RECORDS["bpr_grid"] = bpr_grid
    TRAINING_RECORDS["bpr_gate"] = BPR_GATE
    print(json.dumps(BPR_GATE, indent=1))
    del bpr_model, trained_uf

## 15. Target-aware sequential attention (600-second training budget)

**Architecture** (`target_aware_sequential_attention_v2`)

1. Trainable item embeddings (id 0 is padding; ids are `item_index + 1`) plus **learned positional embeddings** over the right-aligned, positives-only history (most recent item in the last slot).
2. **Target-conditioned multi-head attention.** The candidate item is the query; the history items are keys/values. History relevance is therefore computed *per candidate*.
3. **Gated fusion** of the attended context and the target embedding: `g = sigmoid(W [ctx; t; ctx*t])`, `fused = LayerNorm(g*ctx + (1-g)*t)`.
4. **Nonlinear ranking head** (two-layer GELU MLP on `[fused; ctx*t; t]`) plus a learned per-item bias.
5. **Pairwise loss** `-log sigmoid(s(u, i+) - s(u, i-))` averaged over `ATTENTION_NUM_NEGATIVES` negatives per positive. The positive occurs *after* the history; negatives are never items the user interacted with in training (checked against every training event, not only the history window). Negatives mix uniform and popularity^0.75 sampling.
6. AdamW (no decay on embeddings, biases, norms), gradient clipping, warm-up + cosine learning-rate decay driven by **elapsed time fraction**, mixed precision on CUDA (bf16 where the GPU supports it, otherwise fp16 with a gradient scaler), fixed seeds.

**Time budget.** The timer starts immediately before the first optimizer step (after CUDA synchronisation) and uses `time.monotonic()`. It is never reset per epoch. Validation and logging run *inside* the timed region. Optimisation stops early enough to leave time for one final validation; the measured total (with CUDA synchronisation) is recorded and the notebook fails if it exceeds the budget by more than `ATTENTION_MAX_OVERRUN_SEC`. Initial data preparation and the untrained-baseline validation happen before the timer.

**What the numbers mean.** In-training validation uses 1 positive vs 99 sampled negatives (an *optimistic sampled* metric used only for checkpoint selection). It is not recommendation quality; that is measured later with full-catalogue ranking on the shared candidate pool.

In [ ]:
class TargetAwareSequentialAttention(nn.Module):
    ARCHITECTURE = "target_aware_sequential_attention_v2"

    def __init__(self, n_items, embed_dim=64, n_heads=4, max_history=64, hidden_dim=128, dropout=0.1):
        super().__init__()
        if embed_dim % n_heads:
            raise ValueError("embed_dim must be divisible by n_heads")
        self.n_items, self.embed_dim, self.n_heads = n_items, embed_dim, n_heads
        self.max_history, self.hidden_dim = max_history, hidden_dim
        self.item_embedding = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_history, embed_dim)
        self.item_bias = nn.Embedding(n_items + 1, 1, padding_idx=0)
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.o_proj = nn.Linear(embed_dim, embed_dim)
        self.gate = nn.Linear(embed_dim * 3, embed_dim)
        self.fuse_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim * 3, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Linear(hidden_dim // 2, 1))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.item_embedding.weight, 0.0, 0.05)
        nn.init.normal_(self.position_embedding.weight, 0.0, 0.02)
        nn.init.zeros_(self.item_bias.weight)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

    def forward(self, history, candidates):
        """history [B, L] int64 (0 = padding, ids = item_index + 1, right-aligned);
        candidates [B, C] int64 (ids >= 1). Returns unnormalised scores [B, C]."""
        B, L = history.shape[0], history.shape[1]
        C = candidates.shape[1]
        H, D = self.n_heads, self.embed_dim
        dh = D // H
        mask = history.ne(0)                                                          # [B, L]
        pos_ids = torch.arange(self.max_history, device=history.device)
        h = self.item_embedding(history) + self.position_embedding(pos_ids).unsqueeze(0)   # [B, L, D]
        t = self.item_embedding(candidates)                                            # [B, C, D]
        k = self.k_proj(h).reshape(B, L, H, dh).transpose(1, 2)                        # [B, H, L, dh]
        v = self.v_proj(h).reshape(B, L, H, dh).transpose(1, 2)
        q = self.q_proj(t).reshape(B, C, H, dh).transpose(1, 2)                        # [B, H, C, dh]
        logits = torch.matmul(q, k.transpose(-1, -2)).float() / math.sqrt(dh)           # [B, H, C, L]
        m = mask.unsqueeze(1).unsqueeze(1)
        logits = logits.masked_fill(~m, -1e9)
        w = torch.softmax(logits, dim=-1) * m.to(logits.dtype)
        w = w / (w.sum(dim=-1, keepdim=True) + 1e-9)                                    # all-padding history -> zero context
        ctx = torch.matmul(w.to(v.dtype), v)                                           # [B, H, C, dh]
        ctx = self.o_proj(ctx.transpose(1, 2).reshape(B, C, D))
        gate = torch.sigmoid(self.gate(torch.cat([ctx, t, ctx * t], dim=-1)))
        fused = self.fuse_norm(gate * ctx + (1.0 - gate) * t)
        score = self.head(torch.cat([fused, ctx * t, t], dim=-1)).squeeze(-1)
        return score + self.item_bias(candidates).squeeze(-1)

    def config(self):
        return {"n_items": self.n_items, "embed_dim": self.embed_dim, "n_heads": self.n_heads,
                "max_history": self.max_history, "hidden_dim": self.hidden_dim}


def build_attention_model(cfg, dropout=0.0):
    return TargetAwareSequentialAttention(cfg["n_items"], cfg["embed_dim"], cfg["n_heads"], cfg["max_history"],
                                          cfg["hidden_dim"], dropout)


def count_parameters(model):
    return int(sum(p.numel() for p in model.parameters()))


def assert_finite_state(state, label):
    for name, tensor in state.items():
        if not torch.isfinite(tensor.float()).all():
            raise FloatingPointError(f"non-finite values in {label}:{name}")


def attention_param_groups(model, weight_decay):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        (no_decay if (p.ndim < 2 or "embedding" in name or "norm" in name) else decay).append(p)
    return [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]


def build_attention_sequences(ev_, train_pos_mask_):
    """Positives-only chronological sequences, flattened. Returns ids (+1 offset), per-row sequence start,
    and the flat positions that have at least one earlier item (training examples)."""
    rows = np.flatnonzero(train_pos_mask_)
    users = ev_.user[rows]
    ids = ev_.item[rows].astype(np.int64) + 1
    first = np.r_[True, users[1:] != users[:-1]]
    start_flat = np.flatnonzero(first)
    lens = np.diff(np.r_[start_flat, rows.size])
    row_start = np.repeat(start_flat, lens)
    local = np.arange(rows.size) - row_start
    return {"ids": ids, "users": users.astype(np.int64), "row_start": row_start, "examples": np.flatnonzero(local >= 1)}


def make_attention_batch(flat_idx, seq, max_history):
    """Right-aligned positives-only history ending just before flat_idx, and the positive target."""
    offs = np.arange(-max_history, 0, dtype=np.int64)[None, :]
    pos = flat_idx[:, None] + offs
    valid = pos >= seq["row_start"][flat_idx][:, None]
    hist = np.where(valid, seq["ids"][np.maximum(pos, 0)], 0)
    return hist.astype(np.int64), seq["ids"][flat_idx], seq["users"][flat_idx]


class NegativeSampler:
    """Uniform + popularity^0.75 negatives that are never an item the user interacted with in training."""

    def __init__(self, n_items_, seen_keys_sorted, item_counts, uniform_fraction, power=0.75):
        self.n_items = int(n_items_)
        self.keys = seen_keys_sorted
        w = np.power(np.asarray(item_counts, dtype=np.float64), power)
        self.cdf = np.cumsum(w) / w.sum() if w.sum() > 0 else None
        self.uniform_fraction = uniform_fraction

    def is_seen(self, users, items):
        keys = users.astype(np.int64) * self.n_items + items
        j = np.minimum(np.searchsorted(self.keys, keys), self.keys.size - 1)
        return self.keys[j] == keys

    def sample(self, users, n_neg, rng, extra_exclude=None):
        B = len(users)
        n_uni = n_neg if self.cdf is None else int(round(n_neg * self.uniform_fraction))
        neg = np.empty((B, n_neg), dtype=np.int64)
        neg[:, :n_uni] = rng.integers(0, self.n_items, size=(B, n_uni))
        if n_uni < n_neg:
            neg[:, n_uni:] = np.minimum(np.searchsorted(self.cdf, rng.random((B, n_neg - n_uni)), side="right"), self.n_items - 1)
        users2d = np.broadcast_to(np.asarray(users, dtype=np.int64)[:, None], neg.shape)
        for _ in range(30):
            bad = self.is_seen(users2d, neg)
            if extra_exclude is not None:
                bad |= neg == extra_exclude
            if not bad.any():
                break
            neg[bad] = rng.integers(0, self.n_items, size=int(bad.sum()))
        bad = self.is_seen(users2d, neg)
        if extra_exclude is not None:
            bad |= neg == extra_exclude
        return neg, ~bad


def build_attention_validation(es, sampler, n_users, n_neg, max_history, seed):
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(es.users), size=min(n_users, len(es.users)), replace=False))
    hist = ref.pad_histories([es.pos_items[i] for i in idx], max_history)
    target = es.target[idx]
    neg, ok = sampler.sample(es.users[idx], n_neg, rng, extra_exclude=target[:, None])
    cands = np.concatenate([(target + 1)[:, None], neg + 1], axis=1)
    valid = np.concatenate([np.ones((len(idx), 1), dtype=bool), ok], axis=1)
    return {"hist": hist, "cands": cands, "valid": valid, "users": idx}


@torch.no_grad()
def validate_attention(model, val, device, deadline=None, batch=256):
    """Sampled validation (1 positive vs sampled negatives). Returns None if the deadline passes mid-way."""
    was_training = model.training
    model.eval()
    losses, ranks = [], []
    for b0 in range(0, len(val["hist"]), batch):
        if deadline is not None and time.monotonic() > deadline:
            model.train(was_training)
            return None
        sl = slice(b0, b0 + batch)
        h = torch.as_tensor(val["hist"][sl], device=device)
        c = torch.as_tensor(val["cands"][sl], device=device)
        ok = torch.as_tensor(val["valid"][sl][:, 1:], device=device)
        s = model(h, c).float()
        pos, neg = s[:, :1], s[:, 1:]
        losses.append(float((-F.logsigmoid(pos - neg) * ok).sum() / ok.sum().clamp_min(1)))
        ranks.append((1 + ((neg >= pos) & ok).sum(dim=1)).cpu().numpy())
    model.train(was_training)
    r = np.concatenate(ranks).astype(np.float64)
    return {"val_loss": float(np.mean(losses)), "hr@10": float((r <= 10).mean()),
            "ndcg@10": float(np.where(r <= 10, 1.0 / np.log2(r + 1.0), 0.0).mean()), "mrr": float((1.0 / r).mean())}


def apply_cold_policy(scores, pool, cold_items):
    """Collaborative models carry no signal for items without training positives: set those to the pool minimum."""
    cold = cold_items[pool.items] & pool.valid
    base = np.where(pool.valid & ~cold, scores, np.inf).min(axis=1)
    fill = np.where(np.isfinite(base), base, scores.min(axis=1))
    return np.where(cold, fill[:, None], scores).astype(np.float32)


@torch.no_grad()
def attention_pool_scores(model, es, pool, device, batch_users=64):
    model.eval()
    U, P = pool.items.shape
    hist_all = ref.pad_histories(es.pos_items, model.max_history)
    out = np.zeros((U, P), dtype=np.float32)
    for b0 in range(0, U, batch_users):
        sl = slice(b0, min(U, b0 + batch_users))
        h = torch.as_tensor(hist_all[sl], device=device)
        c = torch.as_tensor(pool.items[sl].astype(np.int64) + 1, device=device)
        out[sl] = model(h, c).float().cpu().numpy()
    return apply_cold_policy(out, pool, MODELS.pop <= 0)

In [ ]:
def train_attention(model, seq, sampler, val, device):
    """Train under a fixed wall-clock budget. Returns (record, best_state, final_state)."""
    budget = ATTENTION_TIME_BUDGET_SEC
    use_cuda = device.type == "cuda"
    amp_dtype, scaler, precision = None, None, "fp32"
    if use_cuda:
        if torch.cuda.get_device_capability()[0] >= 8:
            amp_dtype, precision = torch.bfloat16, "bf16-autocast"
        else:
            amp_dtype, precision = torch.float16, "fp16-autocast+GradScaler"
            try:
                scaler = torch.amp.GradScaler("cuda")
            except (AttributeError, TypeError):          # older torch
                scaler = torch.cuda.amp.GradScaler()
    optimizer = torch.optim.AdamW(attention_param_groups(model, ATTENTION_WEIGHT_DECAY), lr=ATTENTION_LR, betas=(0.9, 0.98))
    examples = seq["examples"]
    n_examples = int(examples.size)
    bs, n_neg = ATTENTION_BATCH_SIZE, ATTENTION_NUM_NEGATIVES
    batches_per_epoch = n_examples // bs
    if batches_per_epoch < 1:
        raise RuntimeError(f"only {n_examples} training examples; need at least one batch of {bs}")
    rng = np.random.default_rng(SEED)

    model.train()
    # Untrained-baseline validation (outside the timed region) also measures validation cost.
    cuda_sync()
    tv = time.monotonic()
    baseline = validate_attention(model, val, device)
    cuda_sync()
    val_cost = time.monotonic() - tv
    reserve = 1.25 * val_cost + 1.0
    print(f"untrained baseline (sampled): {baseline}; one validation pass takes {val_cost:.2f}s; "
          f"final-validation reserve {reserve:.1f}s")

    history = {"train_loss": [], "validation": [], "baseline": baseline}
    best, best_state = None, None
    step, processed, epoch, stop_reason = 0, 0, 0, None
    loss_acc, loss_n = torch.zeros((), device=device), 0
    last_val_wall = None

    def snapshot():
        return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    def run_validation(elapsed_now, deadline):
        nonlocal best, best_state, val_cost
        t_v = time.monotonic()
        res = validate_attention(model, val, device, deadline=deadline)
        cuda_sync()
        dur = time.monotonic() - t_v
        if res is None:
            return None
        val_cost = max(val_cost * 0.5, dur)
        rec = {"step": step, "elapsed_sec": round(elapsed_now, 2), "validation_seconds": round(dur, 2), **res}
        history["validation"].append(rec)
        better = best is None or res["ndcg@10"] > best["ndcg@10"] or (
            res["ndcg@10"] == best["ndcg@10"] and res["val_loss"] < best["val_loss"])
        if better:
            best, best_state = {**res, "step": step, "elapsed_sec": round(elapsed_now, 2)}, snapshot()
        print(f"  step {step:>6} | {elapsed_now:6.1f}s | train_loss {history['train_loss'][-1]['loss'] if history['train_loss'] else float('nan'):.4f} | "
              f"val_loss {res['val_loss']:.4f} | sampled NDCG@10 {res['ndcg@10']:.4f} | HR@10 {res['hr@10']:.4f}{'  *best*' if better else ''}")
        return rec

    # ------------------------------ timed region -------------------------------------------------------------
    cuda_sync()
    t0 = time.monotonic()                          # the timer starts immediately before the first optimizer step
    hard_deadline = t0 + budget
    last_val_wall = t0
    started_utc = datetime.now(timezone.utc).isoformat()
    final_validation_done = False
    lr = ATTENTION_LR
    while stop_reason is None:
        perm = rng.permutation(n_examples)
        epoch += 1
        for b in range(batches_per_epoch):
            now = time.monotonic()
            elapsed = now - t0
            if now >= hard_deadline - reserve:
                stop_reason = "time_budget"
                break
            frac = min(elapsed / budget, 1.0)
            lr = ATTENTION_LR * min(1.0, (step + 1) / ATTENTION_WARMUP_STEPS) * (
                ATTENTION_MIN_LR_RATIO + (1.0 - ATTENTION_MIN_LR_RATIO) * 0.5 * (1.0 + math.cos(math.pi * frac)))
            for g in optimizer.param_groups:
                g["lr"] = lr
            flat = examples[perm[b * bs:(b + 1) * bs]]
            hist_np, pos_np, users_np = make_attention_batch(flat, seq, model.max_history)
            neg_np, ok_np = sampler.sample(users_np, n_neg, rng)
            hist_t = torch.as_tensor(hist_np, device=device)
            cand_t = torch.as_tensor(np.concatenate([pos_np[:, None], neg_np + 1], axis=1), device=device)
            ok_t = torch.as_tensor(ok_np, device=device, dtype=torch.float32)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                scores = model(hist_t, cand_t)
            scores = scores.float()
            loss = (-F.logsigmoid(scores[:, :1] - scores[:, 1:]) * ok_t).sum() / ok_t.sum().clamp_min(1.0)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), ATTENTION_GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), ATTENTION_GRAD_CLIP)
                optimizer.step()
            step += 1
            processed += bs
            loss_acc += loss.detach()
            loss_n += 1
            if step % ATTENTION_LOG_EVERY_STEPS == 0:
                cuda_sync()                                        # CUDA is asynchronous; make the clock honest
                mean_loss = float(loss_acc) / loss_n
                if not math.isfinite(mean_loss):
                    raise FloatingPointError(f"non-finite training loss at step {step}")
                history["train_loss"].append({"step": step, "elapsed_sec": round(time.monotonic() - t0, 2),
                                              "loss": mean_loss, "lr": lr})
                loss_acc, loss_n = torch.zeros((), device=device), 0
                now = time.monotonic()
                if (now - last_val_wall >= ATTENTION_VALIDATION_INTERVAL_SEC
                        and hard_deadline - now > reserve + 1.25 * val_cost):
                    run_validation(now - t0, hard_deadline)
                    last_val_wall = time.monotonic()
    cuda_sync()
    if loss_n:
        mean_loss = float(loss_acc) / loss_n
        if not math.isfinite(mean_loss):
            raise FloatingPointError("non-finite training loss at the end of training")
        history["train_loss"].append({"step": step, "elapsed_sec": round(time.monotonic() - t0, 2), "loss": mean_loss, "lr": lr})
    if not history["validation"] or history["validation"][-1]["step"] != step:
        final_validation_done = run_validation(time.monotonic() - t0, hard_deadline) is not None
    else:
        final_validation_done = True
    cuda_sync()
    elapsed_total = time.monotonic() - t0
    ended_utc = datetime.now(timezone.utc).isoformat()
    # ------------------------------ end of timed region --------------------------------------------------------
    final_state = snapshot()
    if step == 0:
        raise RuntimeError("no optimizer step was executed")
    overrun = max(0.0, elapsed_total - budget)
    record = {
        "architecture": model.ARCHITECTURE, "model_config": model.config(), "n_parameters": count_parameters(model),
        "time_budget_sec": budget, "elapsed_wall_clock_sec": round(elapsed_total, 3), "budget_overrun_sec": round(overrun, 3),
        "budget_respected": overrun <= 0.0, "stop_reason": stop_reason, "final_validation_completed": final_validation_done,
        "timer_started_utc": started_utc, "timer_ended_utc": ended_utc,
        "optimizer_steps": step, "examples_processed": processed, "epochs_completed_fraction": processed / n_examples,
        "training_examples_available": n_examples, "batch_size": bs, "negatives_per_positive": n_neg,
        "precision": precision, "device": str(device),
        "throughput_examples_per_sec": processed / max(elapsed_total, 1e-9),
        "best_checkpoint": best, "final_step": step, "loss_history": history["train_loss"],
        "validation_history": history["validation"], "untrained_baseline_validation": baseline,
        "hyperparameters": {"lr": ATTENTION_LR, "weight_decay": ATTENTION_WEIGHT_DECAY, "grad_clip": ATTENTION_GRAD_CLIP,
                            "warmup_steps": ATTENTION_WARMUP_STEPS, "min_lr_ratio": ATTENTION_MIN_LR_RATIO,
                            "uniform_negative_fraction": ATTENTION_UNIFORM_NEG_FRACTION, "seed": SEED},
        "validation_metric_note": "sampled (1 positive vs 99 sampled negatives); used for checkpoint selection only; "
                                  "NOT recommendation quality",
    }
    return record, (best_state if best_state is not None else final_state), final_state

In [ ]:
ATTENTION_TRAINING = None
with model_stage("attention", critical=True):
    set_seed(SEED)
    cuda_sync()
    t_prep = time.perf_counter()
    seq = build_attention_sequences(ev, train_pos_mask)
    _tm_rows = np.flatnonzero(ev.train_mask)
    seen_keys = np.sort(ev.user[_tm_rows].astype(np.int64) * n_items + ev.item[_tm_rows])      # ALL training events, not just positives
    sampler = NegativeSampler(n_items, seen_keys, item_pos_count, ATTENTION_UNIFORM_NEG_FRACTION)
    val = build_attention_validation(EVAL_VAL, sampler, ATTENTION_VAL_USERS, ATTENTION_VAL_NEGATIVES, ATTENTION_MAX_HISTORY, SEED + 5)
    del _tm_rows
    # Negative-sampler self-test: no sampled negative may be a training interaction of that user.
    _chk_users = seq["users"][seq["examples"][:5000]]
    _neg, _ok = sampler.sample(_chk_users, ATTENTION_NUM_NEGATIVES, np.random.default_rng(1))
    assert _ok.all() and not sampler.is_seen(np.broadcast_to(_chk_users[:, None], _neg.shape), _neg).any()
    print(f"data preparation: {time.perf_counter() - t_prep:.1f}s (outside the timed region)")

    attn_cfg = {"n_items": n_items, "embed_dim": ATTENTION_EMBED_DIM, "n_heads": ATTENTION_HEADS,
                "max_history": ATTENTION_MAX_HISTORY, "hidden_dim": ATTENTION_HIDDEN_DIM}
    attention_model = TargetAwareSequentialAttention(**attn_cfg, dropout=ATTENTION_DROPOUT).to(DEVICE)
    print(f"model parameters: {count_parameters(attention_model):,}; training examples: {seq['examples'].size:,}; "
          f"device {DEVICE}; budget {ATTENTION_TIME_BUDGET_SEC:.0f}s")
    ATTENTION_TRAINING, attn_best_state, attn_final_state = train_attention(attention_model, seq, sampler, val, DEVICE)
    ATTENTION_TRAINING["hardware"] = {k: ENVIRONMENT.get(k) for k in ("gpu_name", "gpu_memory_gb", "gpu_compute_capability",
                                                                         "cpu_count", "ram_total_gb", "torch", "torch_cuda")}
    ATTENTION_TRAINING["peak_gpu_memory_mb"] = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else None
    ATTENTION_TRAINING["peak_rss_mb"] = peak_rss_mb()

    # Fail rather than export a seemingly valid model.
    assert_finite_state(attn_best_state, "best checkpoint")
    assert_finite_state(attn_final_state, "final weights")
    if ATTENTION_TRAINING["budget_overrun_sec"] > ATTENTION_MAX_OVERRUN_SEC:
        raise RuntimeError(f"training exceeded the {ATTENTION_TIME_BUDGET_SEC:.0f}s budget by "
                           f"{ATTENTION_TRAINING['budget_overrun_sec']:.1f}s")
    if ATTENTION_TRAINING["best_checkpoint"] is None:
        raise RuntimeError("no completed validation: cannot select a checkpoint")
    write_json(RUN_DIR / "attention_training.json", ATTENTION_TRAINING)            # persist before any later stage can fail

    t = ATTENTION_TRAINING
    print("\n==================== attention training summary ====================")
    print(f"training duration (wall clock, incl. validation/logging): {t['elapsed_wall_clock_sec']:.1f}s of {t['time_budget_sec']:.0f}s budget "
          f"(overrun {t['budget_overrun_sec']:.2f}s, stop reason: {t['stop_reason']})")
    print(f"training data volume : {t['training_examples_available']:,} positive-sequence examples; "
          f"{t['examples_processed']:,} processed ({t['epochs_completed_fraction']:.2f} epochs)")
    print(f"optimizer steps      : {t['optimizer_steps']:,} at batch size {t['batch_size']} ({t['precision']} on {t['device']})")
    b = t["best_checkpoint"]
    print(f"validation (SAMPLED, 1 vs 99 negatives): best step {b['step']}: NDCG@10 {b['ndcg@10']:.4f}, HR@10 {b['hr@10']:.4f}, "
          f"loss {b['val_loss']:.4f}  | untrained baseline NDCG@10 {t['untrained_baseline_validation']['ndcg@10']:.4f}")
    print("actual recommendation quality: NOT measured yet; see the full-catalogue / shared-pool evaluation below.")
    print("======================================================================")

    attention_model.load_state_dict(attn_best_state)
    attention_model.eval()
    del seq, sampler, val
    free_memory()

## 16. Validation evaluation of every model

All models are scored for the same validation users with the same history and exclusions.

* **Protocol A (full catalogue):** random, popularity, content, ItemKNN, ALS (fold-in), BPR (fold-in).
* **Protocol B (shared candidate pool):** every model *and* the attention model score the same per-user pool. A target outside the pool is a miss for every model in this protocol, so the pool-recall ceiling is reported next to it.

In [ ]:
def available(name):
    return STAGES.get(name, {}).get("status") == "available"


FULL_MODELS = ["random", "popularity"] + [m for m in ("content", "itemknn", "als", "bpr") if available(m)]
GENERATORS = ["popularity"] + [m for m in ("content", "itemknn", "als") if available(m)] + (["bpr"] if BPR_PRODUCTION_ENABLED and available("bpr") else [])
HYBRID_MODELS = GENERATORS + ["attention"]
print("generators:", GENERATORS, "| hybrid models:", HYBRID_MODELS, "| BPR production-enabled:", BPR_PRODUCTION_ENABLED)

t0 = time.perf_counter()
val_full, val_pool = run_split_evaluation(MODELS, EVAL_VAL, FULL_MODELS, GENERATORS, CANDIDATE_POOL_PER_SOURCE, desc="validation")
val_pool.scores["attention"] = attention_pool_scores(attention_model, EVAL_VAL, val_pool, DEVICE)
POOL_FULL_MODELS = FULL_MODELS + ["attention"]
print(f"validation evaluation took {time.perf_counter() - t0:.0f}s; pool size mean {val_pool.sizes.mean():.0f}, max {val_pool.sizes.max()}")


def pool_recall_report(full, pool, es):
    out = {"union_pool_recall": float((pool.target_col >= 0).mean()), "mean_pool_size": float(pool.sizes.mean()),
           "per_source_candidate_recall": {}}
    for g in pool.generators:
        out["per_source_candidate_recall"][g] = float((full[g]["ranks"] <= CANDIDATE_POOL_PER_SOURCE).mean())
    return out


val_reports_full = {m: full_report(val_full[m], EVAL_VAL) for m in FULL_MODELS}
val_pool_results = {m: pool_rank(val_pool.scores[m], val_pool) for m in POOL_FULL_MODELS}
val_reports_pool = {m: full_report(val_pool_results[m], EVAL_VAL) for m in POOL_FULL_MODELS}
VAL_POOL_RECALL = pool_recall_report(val_full, val_pool, EVAL_VAL)
print(json.dumps(VAL_POOL_RECALL, indent=1))
print("\nProtocol A (full catalogue), validation:")
display(results_frame(val_reports_full).round(4))
print("Protocol B (shared candidate pool), validation:")
display(results_frame(val_reports_pool).round(4))

## 17. Hybrid: normalisation, validation-tuned weights, ablation, MMR

1. **Candidate generation:** shared pool above (BPR only if production-enabled).
2. **Normalisation per user over the pool:** min-max, z-score or rank. Raw scores of popularity, cosine, dot-product and neural logits are not comparable, so they are never added directly.
3. **Weights:** random search on the probability simplex (`HYBRID_WEIGHT_TRIALS` Dirichlet draws plus each single model and equal weights), objective = validation NDCG@10. This is a search, **not** a proof of optimality; weights are rounded and renormalised. Reciprocal-rank fusion with equal weights is reported as a simple baseline.
4. **Ablation:** each model is removed in turn (remaining weights renormalised) and validation NDCG@10 re-measured.
5. **MMR:** `lambda * relevance - (1 - lambda) * max content-similarity to already selected items`, over the top `MMR_DEPTH` fused candidates. `lambda` is chosen as the **most diverse** value whose validation NDCG@10 stays within `MMR_MAX_RELATIVE_NDCG_LOSS` of the no-MMR hybrid. The trade-off is reported, not hidden.
6. **Fallbacks** (implemented in the reference recommender and tested): a model that is unavailable is dropped and weights are renormalised; a user without positive history gets popularity; if no scoring model remains, popularity is served.

In [ ]:
def fast_ndcg10(fused, pool):
    s = np.where(pool.valid, fused, -np.inf)
    U = s.shape[0]
    tc = pool.target_col
    ts = s[np.arange(U), np.maximum(tc, 0)]
    ranks = (s >= ts[:, None]).sum(axis=1)
    hit = (tc >= 0) & (ranks <= 10)
    return float(np.where(hit, 1.0 / np.log2(ranks + 1.0), 0.0).mean())


def hybrid_rerank(fused, pool, target, lam, depth):
    U = fused.shape[0]
    ranks = np.full(U, np.inf)
    top = np.full((U, KMAX), -1, dtype=np.int32)
    for u in range(U):
        v = pool.valid[u]
        items = pool.items[u][v].astype(np.int64)
        pos = ref.mmr_rerank(fused[u][v], items, EVALCTX.content, KMAX, lam, depth)
        lst = items[pos]
        top[u, :lst.size] = lst
        hit = np.flatnonzero(lst == target[u])
        if hit.size:
            ranks[u] = hit[0] + 1
    return ranks_to_result(ranks, top)


def normalized_pool_scores(pool, models, method):
    return {m: ref.normalize_scores(pool.scores[m], pool.valid, method) for m in models}


def round_weights(w, models):
    w = np.asarray(w, dtype=np.float64)
    w = np.round(w / w.sum(), 4)
    w[np.argmax(w)] += 1.0 - w.sum()
    return {m: float(round(x, 4)) for m, x in zip(models, w)}


rng = np.random.default_rng(SEED + 9)
k_models = len(HYBRID_MODELS)
weight_candidates = [np.eye(k_models)[i] for i in range(k_models)] + [np.full(k_models, 1.0 / k_models)] \
                    + list(rng.dirichlet(np.ones(k_models), size=HYBRID_WEIGHT_TRIALS))
search_log, best_h = {}, None
for method in HYBRID_NORMALIZATIONS:
    normed = normalized_pool_scores(val_pool, HYBRID_MODELS, method)
    scored = []
    for w in weight_candidates:
        fused = ref.fuse_scores(normed, dict(zip(HYBRID_MODELS, w)))
        scored.append((fast_ndcg10(fused, val_pool), w))
    scored.sort(key=lambda x: -x[0])
    search_log[method] = {"best_ndcg@10": scored[0][0], "best_weights": round_weights(scored[0][1], HYBRID_MODELS),
                          "equal_weight_ndcg@10": fast_ndcg10(ref.fuse_scores(normed, dict(zip(HYBRID_MODELS, np.full(k_models, 1.0 / k_models)))), val_pool),
                          "trials": len(weight_candidates)}
    if best_h is None or scored[0][0] > best_h[0]:
        best_h = (scored[0][0], method, scored[0][1])
HYB_NDCG, HYB_METHOD, _w = best_h
HYB_WEIGHTS = round_weights(_w, HYBRID_MODELS)
_normed = normalized_pool_scores(val_pool, HYBRID_MODELS, HYB_METHOD)
HYB_FUSED_VAL = ref.fuse_scores(_normed, HYB_WEIGHTS)
print(f"selected normalisation: {HYB_METHOD}; weights: {HYB_WEIGHTS}; validation NDCG@10 (pool) = {fast_ndcg10(HYB_FUSED_VAL, val_pool):.4f}")

# ----- baselines and ablation -------------------------------------------------------------------------------------
rrf_normed = normalized_pool_scores(val_pool, HYBRID_MODELS, "rrf")
rrf_equal = ref.fuse_scores(rrf_normed, {m: 1.0 / k_models for m in HYBRID_MODELS})
HYBRID_COMPARISON = {"hybrid_tuned": fast_ndcg10(HYB_FUSED_VAL, val_pool),
                     "rrf_equal_weights_baseline": fast_ndcg10(rrf_equal, val_pool)}
for m in HYBRID_MODELS:
    HYBRID_COMPARISON[f"single:{m}"] = fast_ndcg10(val_pool.scores[m], val_pool)
ablation = {}
for drop in HYBRID_MODELS:
    rest = {m: w for m, w in HYB_WEIGHTS.items() if m != drop}
    if sum(rest.values()) > 0:
        ablation[f"without_{drop}"] = fast_ndcg10(ref.fuse_scores(_normed, rest), val_pool)
HYBRID_COMPARISON["ablation"] = ablation
print(json.dumps(HYBRID_COMPARISON, indent=1))
del rrf_normed, rrf_equal

# ----- MMR trade-off ---------------------------------------------------------------------------------------------------
mmr_table, mmr_results = [], {}
for lam in MMR_LAMBDA_GRID:
    res = hybrid_rerank(HYB_FUSED_VAL, val_pool, EVAL_VAL.target, lam, MMR_DEPTH)
    s = summarize_result(res)
    mmr_results[lam] = res
    mmr_table.append({"lambda": lam, "ndcg@10": s["ndcg@10"], "recall@10": s["recall@10"], "diversity@10": s["diversity@10"],
                      "coverage@10": s["coverage@10"], "novelty@10": s["novelty@10"]})
mmr_df = pd.DataFrame(mmr_table)
display(mmr_df.round(4))
base_ndcg = mmr_df.loc[mmr_df["lambda"] == 1.0, "ndcg@10"].iloc[0]
ok = mmr_df[mmr_df["ndcg@10"] >= base_ndcg * (1.0 - MMR_MAX_RELATIVE_NDCG_LOSS)]
MMR_LAMBDA = float(ok.sort_values(["diversity@10", "lambda"], ascending=[False, True]).iloc[0]["lambda"])
print(f"selected MMR lambda = {MMR_LAMBDA} (most diverse within {MMR_MAX_RELATIVE_NDCG_LOSS:.0%} NDCG@10 loss of lambda=1.0)")
val_hybrid_results = {"hybrid_no_mmr": mmr_results[1.0], "hybrid": mmr_results[MMR_LAMBDA]}
val_reports_pool["hybrid_no_mmr"] = full_report(val_hybrid_results["hybrid_no_mmr"], EVAL_VAL)
val_reports_pool["hybrid"] = full_report(val_hybrid_results["hybrid"], EVAL_VAL)
print("Validation, shared pool, incl. hybrid:")
display(results_frame(val_reports_pool).round(4))
TRAINING_RECORDS["hybrid_search"] = search_log
TRAINING_RECORDS["hybrid_comparison"] = HYBRID_COMPARISON
TRAINING_RECORDS["mmr_table"] = mmr_table
free_memory()

## 18. Final temporal TEST evaluation (run once)

Nothing is tuned after this point. The configuration selected on validation (hyperparameters, normalisation, weights, lambda) is applied to the test targets. At test time each user's history contains every event before the test target; fitted item-level parameters are not refit. The cell refuses to run twice.

In [ ]:
if globals().get("_TEST_EVALUATED"):
    raise RuntimeError("the test split has already been evaluated; do not iterate on it")
_TEST_EVALUATED = True

t0 = time.perf_counter()
test_full, test_pool = run_split_evaluation(MODELS, EVAL_TEST, FULL_MODELS, GENERATORS, CANDIDATE_POOL_PER_SOURCE, desc="test")
test_pool.scores["attention"] = attention_pool_scores(attention_model, EVAL_TEST, test_pool, DEVICE)
_t_normed = normalized_pool_scores(test_pool, HYBRID_MODELS, HYB_METHOD)
test_fused = ref.fuse_scores(_t_normed, HYB_WEIGHTS)
test_pool_results = {m: pool_rank(test_pool.scores[m], test_pool) for m in POOL_FULL_MODELS}
test_pool_results["hybrid_no_mmr"] = hybrid_rerank(test_fused, test_pool, EVAL_TEST.target, 1.0, MMR_DEPTH)
test_pool_results["hybrid"] = hybrid_rerank(test_fused, test_pool, EVAL_TEST.target, MMR_LAMBDA, MMR_DEPTH)

test_reports_full = {m: full_report(test_full[m], EVAL_TEST) for m in FULL_MODELS}
test_reports_pool = {m: full_report(r, EVAL_TEST) for m, r in test_pool_results.items()}
TEST_POOL_RECALL = pool_recall_report(test_full, test_pool, EVAL_TEST)
print(f"test evaluation took {time.perf_counter() - t0:.0f}s")
print(json.dumps(TEST_POOL_RECALL, indent=1))
print("\nTEST, Protocol A (full catalogue):")
display(results_frame(test_reports_full).round(4))
print("TEST, Protocol B (shared candidate pool):")
display(results_frame(test_reports_pool).round(4))

print("\nTEST, shared pool: NDCG@10 and Recall@10 by user-history cohort")
display(pd.DataFrame({m: {c: r["ndcg@10"] for c, r in rep["cohorts"].items()} for m, rep in test_reports_pool.items()}).T.round(4))
print("TEST, shared pool: Recall@10 by target popularity band")
display(pd.DataFrame({m: {b: r["recall@10"] for b, r in rep["target_bands"].items()} for m, rep in test_reports_pool.items()}).T.round(4))
print("A lower training loss never appears in these tables: models are compared only on held-out ranking metrics.")

## 19. Export the release

Files are written to `backend/artifacts/models/releases/<release_id>/` (flat layout, only files a model needs). Formats: float32 `.npy`, CSR `.npz`, JSON, Safetensors, ONNX; nothing is pickled. The release is first written as a **draft** manifest; it is marked `complete` only after every validation below passes.

Not exported: raw ratings, MovieLens user ids, user factors, the catalogue CSV, holdout rows, training matrices. `item_provider_ids.npy` is an alignment contract (row *i* of every item-indexed artifact is provider id *i*), not a catalogue.

In [ ]:
def describe_file(path):
    info = {"sha256": ref.file_sha256(path), "bytes": path.stat().st_size}
    if path.suffix == ".npy":
        a = np.load(path, mmap_mode="r", allow_pickle=False)
        info.update(dtype=str(a.dtype), shape=list(a.shape))
    elif path.suffix == ".npz":
        m = sp.load_npz(path)
        info.update(dtype=str(m.dtype), shape=list(m.shape), nnz=int(m.nnz), format="csr")
    return info


def export_onnx(model_cpu, path, max_history, opset):
    model_cpu.eval()
    hist = torch.zeros(2, max_history, dtype=torch.long)
    hist[0, -3:] = torch.tensor([5, 6, 7])
    hist[1, -max_history // 2:] = 3
    cands = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.long)
    kwargs = dict(input_names=["history", "candidates"], output_names=["scores"], opset_version=opset,
                  dynamic_axes={"history": {0: "batch"}, "candidates": {0: "batch", 1: "num_candidates"},
                                "scores": {0: "batch", 1: "num_candidates"}}, do_constant_folding=True)
    with torch.no_grad():
        try:
            torch.onnx.export(model_cpu, (hist, cands), str(path), dynamo=False, **kwargs)
        except TypeError:                       # older torch without the dynamo argument
            torch.onnx.export(model_cpu, (hist, cands), str(path), **kwargs)
    import onnx
    onnx.checker.check_model(onnx.load(str(path)))
    if Path(str(path) + ".data").exists():
        raise RuntimeError("ONNX export produced external data; a single self-contained file is required")


def attention_parity(model_cpu, onnx_path, max_history, n_items_, use_gpu, atol):
    """Max |PyTorch - ONNX Runtime| over padded, partially padded, full and empty histories."""
    sess = ref.make_ort_session(onnx_path, use_gpu=use_gpu)
    provider = sess.get_providers()[0]
    rng = np.random.default_rng(SEED + 11)
    L = max_history
    cases = [(1, 5, [0]), (4, 50, [0, 1, L // 2, L]), (8, 400, [3, 7, L, L, 10, L - 1, 2, 0]), (2, 1, [L, 2])]
    worst = 0.0
    for B, C, lens in cases:
        hist = ref.pad_histories([rng.integers(0, n_items_, size=l) for l in lens], L)
        cands = rng.integers(1, n_items_ + 1, size=(B, C)).astype(np.int64)
        with torch.no_grad():
            expected = model_cpu(torch.as_tensor(hist), torch.as_tensor(cands)).numpy()
        got = ref.attention_scores_onnx(sess, hist, cands)
        if got.shape != expected.shape or not np.isfinite(got).all():
            raise AssertionError(f"ONNX output invalid for case {(B, C, lens)}: shape {got.shape}")
        worst = max(worst, float(np.abs(got - expected).max()))
    if worst > atol:
        raise AssertionError(f"ONNX vs PyTorch max abs difference {worst:.3e} exceeds tolerance {atol:.1e} (provider {provider})")
    return {"max_abs_diff": worst, "tolerance": atol, "actual_provider": provider, "cases": len(cases)}


def run_artifact_tests(release_dir, strict_load):
    """Validate a release directory and exercise CPU (and, when really available, GPU) inference.
    strict_load=True also verifies checksums and requires a complete manifest."""
    import onnx, onnxruntime
    from safetensors.torch import load_file
    release_dir = Path(release_dir)
    out = {}
    rel = ref.load_release(release_dir, verify_checksums=strict_load, require_complete=strict_load)
    assert np.array_equal(rel.item_provider_ids, source_movie_ids), "item_provider_ids misaligned with the training index"
    out["structure"] = "arrays, dtypes, shapes, finiteness, sparse validity, index alignment: OK"

    cfg = rel.configs["attention"]
    sd = load_file(str(release_dir / "attention.safetensors"))
    model_cpu = build_attention_model(cfg).eval()
    expected_shapes = {k: tuple(v.shape) for k, v in model_cpu.state_dict().items()}
    assert {k: tuple(v.shape) for k, v in sd.items()} == expected_shapes, "attention config does not match safetensors shapes"
    model_cpu.load_state_dict(sd, strict=True)
    for k, v in sd.items():
        assert torch.isfinite(v.float()).all(), f"non-finite attention weight {k}"
    onnx.checker.check_model(onnx.load(str(release_dir / "attention.onnx")))
    out["onnx_graph"] = "valid"
    out["onnx_cpu_parity"] = attention_parity(model_cpu, release_dir / "attention.onnx", cfg["max_history"], rel.n_items, False, ATTENTION_ONNX_ATOL_CPU)

    ort_has_cuda = "CUDAExecutionProvider" in onnxruntime.get_available_providers()
    if torch.cuda.is_available() and ort_has_cuda:
        gpu = attention_parity(model_cpu, release_dir / "attention.onnx", cfg["max_history"], rel.n_items, True, ATTENTION_ONNX_ATOL_GPU)
        out["onnx_gpu_parity"] = gpu if gpu["actual_provider"] == "CUDAExecutionProvider" else {
            "status": "NOT TESTED on GPU: ONNX Runtime fell back to " + gpu["actual_provider"], **gpu}
    else:
        out["onnx_gpu_parity"] = {"status": f"skipped (torch CUDA: {torch.cuda.is_available()}, ORT CUDA provider: {ort_has_cuda})"}
    if torch.cuda.is_available():
        m_gpu = build_attention_model(cfg).eval().to("cuda")
        m_gpu.load_state_dict(sd, strict=True)
        h = ref.pad_histories([[1, 2, 3], []], cfg["max_history"])
        c = np.array([[1, 2, 3, 4], [5, 6, 7, 8]], dtype=np.int64)
        with torch.no_grad():
            a = model_cpu(torch.as_tensor(h), torch.as_tensor(c)).numpy()
            b = m_gpu(torch.as_tensor(h).cuda(), torch.as_tensor(c).cuda()).float().cpu().numpy()
        diff = float(np.abs(a - b).max())
        assert diff < ATTENTION_ONNX_ATOL_GPU, f"PyTorch CUDA vs CPU difference {diff}"
        out["torch_cuda_vs_cpu_max_abs_diff"] = diff
        del m_gpu

    # ----- ALS fold-in for an unseen user: normal-equation residual ------------------------------------------
    rng = np.random.default_rng(SEED + 12)
    hist_items = rng.choice(np.flatnonzero(rel.arrays["popularity"] > 0), size=25, replace=False)
    hist_rel = ref.rating_to_relevance(rng.choice([3.5, 4.0, 4.5, 5.0], size=25))
    ac = rel.configs["als"]
    x = ref.als_fold_in(rel.arrays["als"], rel.arrays["als_gram"], hist_items, hist_rel, ac["confidence_alpha"], ac["regularization"])
    Y = rel.arrays["als"].astype(np.float64)
    c_all = np.ones(rel.n_items)
    c_all[hist_items] = 1 + ac["confidence_alpha"] * hist_rel
    p = np.zeros(rel.n_items)
    p[hist_items] = 1
    A = Y.T @ (Y * c_all[:, None]) + ac["regularization"] * np.eye(Y.shape[1])
    resid = np.linalg.norm(A @ x - Y.T @ (c_all * p)) / np.linalg.norm(Y.T @ (c_all * p))
    assert np.isfinite(x).all() and resid < 1e-4, f"ALS fold-in residual {resid}"
    out["als_fold_in_unseen_user_relative_residual"] = float(resid)

    # ----- content model: new catalogue item that is not in the trained matrix ----------------------------------
    cc = rel.configs["content"]
    idf = rel.arrays["content_idf"]
    sample_items = rng.choice(rel.n_items, size=300, replace=False)
    toks = [ref.content_tokens(movies_df["title"].iloc[i], movies_df["genres"].iloc[i]) for i in sample_items]
    rebuilt = ref.content_vectors(toks, cc["vocabulary"], idf, cc["block_weights"])
    assert np.abs((rebuilt - rel.arrays["content"][sample_items]).toarray()).max() < 1e-6, "JSON vocabulary/IDF do not reproduce stored vectors"
    new_vec = ref.content_vectors([ref.content_tokens("Orbital Heist Chronicles (2031)", "Sci-Fi|Action|Crime")], cc["vocabulary"], idf, cc["block_weights"])
    assert new_vec.nnz > 0 and np.isfinite(new_vec.data).all() and abs(float(np.sqrt(new_vec.multiply(new_vec).sum())) - 1.0) < 1e-4
    sims = np.asarray((new_vec @ rel.arrays["content"].T).todense()).ravel()
    top = ref.topk_indices(sims, 10)
    shared = [len(set(movies_df["genres"].iloc[i].split("|")) & {"Sci-Fi", "Action", "Crime"}) for i in top]
    assert sum(s > 0 for s in shared) >= 7, "recommendations for a brand-new item do not share its genres"
    out["content_new_item"] = "vector built from JSON vocabulary/IDF only; recommendations share genres: OK"

    # ----- end-to-end recommender: CPU, unseen user, ablations, fallbacks, determinism ------------------------
    rec = ref.CineRankRecommender(release_dir, use_gpu=False, verify_checksums=strict_load, require_complete=strict_load)
    history = [(int(i), float(r)) for i, r in zip(hist_items, rng.choice([4.0, 5.0], size=25))] + \
              [(int(i), 2.0) for i in rng.choice(rel.n_items, size=3, replace=False)]
    seen = {i for i, _ in history}
    ablations = [(), ("attention",), ("als",), ("itemknn", "content"), ("attention", "als", "itemknn", "content", "bpr")]
    for disabled in ablations:
        r2 = ref.CineRankRecommender(release_dir, disabled_models=disabled, verify_checksums=strict_load, require_complete=strict_load)
        res = r2.recommend(history, k=20)
        items = res["items"]
        assert items.size == 20 and np.unique(items).size == 20 and not (set(items.tolist()) & seen), f"bad list for {disabled}"
        assert np.isfinite(res["scores"]).all() and items.min() >= 0 and items.max() < rel.n_items
    cold = rec.recommend([], k=20)
    assert cold["fallback"] == "cold_start_popularity" and cold["items"].size == 20
    only_negative = rec.recommend([(int(i), 1.5) for i in hist_items[:5]], k=20)
    assert only_negative["fallback"] == "cold_start_popularity"
    assert np.array_equal(rec.recommend(history, k=20)["items"], rec.recommend(history, k=20)["items"]), "inference is not deterministic"
    out["cpu_inference"] = {"ablations_tested": [list(a) for a in ablations], "attention_provider": rec.attention_provider,
                            "cold_start": "popularity fallback OK", "deterministic": True}

    # ----- consistency between the batched evaluation pipeline and the serving code ------------------------------
    overlaps = []
    for u in list(range(0, len(EVAL_VAL.users), max(1, len(EVAL_VAL.users) // 20)))[:20]:
        hist = [(int(i), float(r)) for i, r in zip(EVAL_VAL.hist_items[u], EVAL_VAL.hist_ratings[u])]
        served = rec.recommend(hist, k=KMAX)["items"][:10]
        evaluated = val_hybrid_results["hybrid"]["topk"][u, :10]
        overlaps.append(len(set(served.tolist()) & set(evaluated.tolist())) / 10.0)
    out["serving_vs_evaluation_top10_overlap_mean"] = float(np.mean(overlaps))
    assert np.mean(overlaps) >= 0.9, f"serving code and evaluation pipeline disagree (overlap {np.mean(overlaps):.2f})"
    return out

In [ ]:
with model_stage("export", critical=True):
    from safetensors.torch import save_file
    RD = RELEASE_DIR
    np.save(RD / "item_provider_ids.npy", source_movie_ids.astype(np.int64))
    np.save(RD / "popularity_scores.npy", popularity_scores.astype(np.float32))
    write_json(RD / "popularity_config.json", POP_CONFIG)

    if available("content"):
        sp.save_npz(RD / "content_item_vectors.npz", content_vectors.astype(np.float32).tocsr(), compressed=True)
        np.save(RD / "content_idf.npy", content_idf.astype(np.float32))
        write_json(RD / "content_config.json", {
            "schema_version": 1, "model": "content_tfidf", "n_items": n_items, "n_features": len(content_vocab),
            "vocabulary": content_vocab, "idf_file": "content_idf.npy", "block_weights": CONTENT_BLOCK_WEIGHTS,
            "min_df": CONTENT_MIN_DF, "max_features": CONTENT_MAX_FEATURES, "term_frequency": "binary",
            "idf_formula": "ln((1+N)/(1+df)) + 1", "normalization": "l2 per item vector",
            "tokenizer": {"normalize": "NFKD -> ASCII -> lower-case", "word_regex": "[a-z0-9]+", "stopwords": sorted(ref.CONTENT_STOPWORDS),
                          "min_word_length": "2, digits allowed at length 1", "title_year": "trailing (YYYY) removed from title and parsed",
                          "tokens": {"t:": "title word", "tb:": "title bigram of consecutive non-stopword words",
                                     "g:": "genre, non-alphanumerics -> _", "d:": "release decade, e.g. d:1990s"},
                          "reference_implementation": "reference_inference.content_tokens / content_vectors"},
            "new_item_contract": "tokenize title+genres with content_tokens, drop tokens absent from the vocabulary, weight by "
                                 "block_weight*idf, L2-normalise. Items need no interaction history.",
            "user_profile": "L2-normalised sum of positive-history item vectors; score = cosine with every item vector"})
    if available("itemknn"):
        sp.save_npz(RD / "itemknn_topk.npz", itemknn_topk.astype(np.float32).tocsr(), compressed=True)
        write_json(RD / "itemknn_config.json", {
            "schema_version": 1, "model": "itemknn_topk", "n_items": n_items, "k": int(SELECTED["itemknn"]["k"]),
            "weighting": SELECTED["itemknn"]["weighting"], "shrinkage": SELECTED["itemknn"]["shrinkage"],
            "min_item_support": KNN_MIN_ITEM_SUPPORT, "similarity": "cosine over positive interactions, shrunk by n_ij/(n_ij+shrinkage)",
            "orientation": "row j lists the neighbours of item j; entries are similarities >= 0; no self-similarity",
            "scoring": "score_i = sum over positive-history items j of S[j, i]", "nnz": int(itemknn_topk.nnz),
            "validation": {k: v for k, v in SELECTED["itemknn"].items() if k.startswith(("ndcg", "recall", "mrr"))}})
    if available("als"):
        np.save(RD / "als_item_factors.npy", MODELS.als_Y.astype(np.float32))
        write_json(RD / "als_config.json", {
            "schema_version": 1, "model": "implicit_als", "n_items": n_items, "factors": int(SELECTED["als"]["factors"]),
            "item_factor_columns": int(MODELS.als_Y.shape[1]), "regularization": float(MODELS.als_reg),
            "confidence_alpha": IMPLICIT_CONFIDENCE_ALPHA, "iterations": int(SELECTED["als"]["iterations"]),
            "confidence": "c = 1 + alpha * relevance(rating) over POSITIVE interactions only; relevance = clip((rating-0.5)/4.5, 0, 1)",
            "fold_in": "x_u = (Y^T Y + sum_i (c_ui-1) y_i y_i^T + reg I)^-1 sum_i c_ui y_i ; score = Y x_u",
            "user_factors_exported": False, "positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
            "validation": {k: v for k, v in SELECTED["als"].items() if k.startswith(("ndcg", "recall", "mrr"))},
            "fold_in_check": TRAINING_RECORDS.get("als_foldin_check")})
    if available("bpr"):
        np.save(RD / "bpr_item_factors.npy", MODELS.bpr_Y.astype(np.float32))
        write_json(RD / "bpr_config.json", {
            "schema_version": 1, "model": "implicit_bpr", "n_items": n_items, "factors": BPR_FACTORS,
            "item_factor_columns": int(MODELS.bpr_Y.shape[1]), "item_bias_column": -1,
            "layout": "[:, :-1] latent factors, [:, -1] item bias; score = u . v_i + b_i",
            "fold_in": {**MODELS.bpr_fold, "method": "Adam on the pairwise BPR loss with item factors frozen; negatives uniform, never from history; "
                                                      "seed = seed + sum(positive item indices) % 1000003"},
            "production_enabled": BPR_PRODUCTION_ENABLED, "gate": TRAINING_RECORDS.get("bpr_gate"),
            "user_factors_exported": False,
            "warning": "Disabled in production unless the validation gate passed. Averaging item factors is not equivalent to BPR inference."})

    best_cfg = ATTENTION_TRAINING["model_config"]
    save_file({k: v.contiguous() for k, v in attn_best_state.items()}, str(RD / "attention.safetensors"),
              metadata={"format": "pt", "architecture": TargetAwareSequentialAttention.ARCHITECTURE, "release_id": RELEASE_ID})
    _m = build_attention_model(best_cfg).eval()
    from safetensors.torch import load_file as _load
    _m.load_state_dict(_load(str(RD / "attention.safetensors")), strict=True)
    export_onnx(_m, RD / "attention.onnx", best_cfg["max_history"], ATTENTION_ONNX_OPSET)
    write_json(RD / "attention_config.json", {
        "schema_version": 1, "model": "target_aware_sequential_attention", "architecture": TargetAwareSequentialAttention.ARCHITECTURE,
        **best_cfg, "dropout_at_inference": 0.0, "id_offset": 1, "pad_id": 0,
        "history": "last max_history POSITIVE items, chronological, right-aligned, left-padded with 0; ids = item_index + 1",
        "inputs": {"history": "int64 [batch, max_history] (fixed length)", "candidates": "int64 [batch, num_candidates] (ids >= 1)"},
        "outputs": {"scores": "float32 [batch, num_candidates]; unnormalised logits, comparable only within one request"},
        "weights_file": "attention.safetensors", "onnx_file": "attention.onnx",
        "onnx": {"opset": ATTENTION_ONNX_OPSET, "dynamic_axes": ["batch", "num_candidates"], "fixed_axes": ["max_history"]},
        "onnx_parity_tolerance": {"cpu_abs": ATTENTION_ONNX_ATOL_CPU, "gpu_abs": ATTENTION_ONNX_ATOL_GPU,
                                  "basis": "fp32 inference; accumulation-order differences ~1e-6, CUDA kernels may use TF32"},
        "cold_item_policy": "score for items with popularity_scores == 0 is replaced by the candidate-pool minimum",
        "training_summary": {k: ATTENTION_TRAINING[k] for k in ("time_budget_sec", "elapsed_wall_clock_sec", "budget_respected", "optimizer_steps",
                                                              "examples_processed", "training_examples_available", "precision", "device")},
        "checkpoint": "best validation checkpoint (sampled NDCG@10); final-step weights are kept in the training run directory",
        "sampled_validation_best": ATTENTION_TRAINING["best_checkpoint"]})
    del _m

    HYBRID_CONFIG = {
        "schema_version": 1, "config_id": f"{RELEASE_ID}-hybrid", "algorithm": "normalized_weighted_fusion_with_mmr",
        "feedback": {"positive_rating_threshold": POSITIVE_RATING_THRESHOLD, "negative_rating_threshold": NEGATIVE_RATING_THRESHOLD,
                     "relevance_mapping": "clip((rating - 0.5) / 4.5, 0, 1)"},
        "candidate_generation": {"sources": GENERATORS, "per_source": CANDIDATE_POOL_PER_SOURCE,
                                 "pool": "union of each source's top-N, excluding every item in the user's history"},
        "scoring_models": [m for m in HYBRID_MODELS if HYB_WEIGHTS[m] > 0], "normalization": HYB_METHOD, "weights": HYB_WEIGHTS,
        "cold_item_policy": {"models": list(COLD_ITEM_POLICY_MODELS), "rule": "popularity_scores == 0",
                             "behavior": "score := minimum of the candidate pool before normalisation"},
        "mmr": {"enabled": bool(available("content") and MMR_LAMBDA < 1.0), "lambda": MMR_LAMBDA, "candidate_depth": MMR_DEPTH,
                "similarity": "cosine of content_item_vectors rows"},
        "fallback": {"cold_start_user": "popularity", "model_unavailable": "drop the model, renormalise remaining weights per request",
                     "no_scoring_model": "popularity", "chain": ["hybrid", "als", "itemknn", "content", "popularity"]},
        "tuning": {"method": f"random search over the weight simplex ({len(weight_candidates)} candidates per normalisation) maximising validation NDCG@10 "
                             "on the shared candidate pool; this is a search result, not a proven optimum",
                   "validation_ndcg@10_pool": float(fast_ndcg10(HYB_FUSED_VAL, val_pool)), "mmr_rule": f"most diverse lambda within {MMR_MAX_RELATIVE_NDCG_LOSS:.0%} NDCG@10 loss"},
    }
    write_json(RD / "hybrid_config.json", HYBRID_CONFIG)


def write_manifest(status):
    files = {p.name: describe_file(p) for p in sorted(RELEASE_DIR.iterdir()) if p.is_file() and p.name not in ("manifest.json", "SHA256SUMS")}
    models = {}
    for name, fnames in ref.MODEL_FILES.items():
        ok = available(name) if name not in ("popularity", "hybrid") else True
        models[name] = {"status": "available" if ok else "unavailable", "files": fnames if ok else [],
                        "production_enabled": bool(ok and (BPR_PRODUCTION_ENABLED if name == "bpr" else True))}
        if not ok:
            models[name]["reason"] = STAGES.get(name, {}).get("error", "not trained")
    write_json(RELEASE_DIR / "manifest.json", {
        "schema_version": ref.SCHEMA_VERSION, "release_id": RELEASE_ID, "training_run_id": RUN_ID, "status": status,
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "provider": "movielens", "item_id_file": ref.ITEM_ID_FILE,
        "n_items": n_items, "models": models, "files": files,
        "dataset": {k: DATASET_INFO[k] for k in ("name", "url", "directory", "archive_sha256")},
        "training_cohort": {"users": DATA_SUMMARY["n_users"], "sampling": SAMPLING_INFO["sampling"], "seed": SEED,
                            "note": "a sample of MovieLens users; not the CineRank population"},
        "feedback_semantics": {"positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
                               "negative_rating_threshold": NEGATIVE_RATING_THRESHOLD,
                               "missing_observations": "not dislikes"},
        "attention_training": {k: ATTENTION_TRAINING[k] for k in ("time_budget_sec", "elapsed_wall_clock_sec", "optimizer_steps", "budget_respected")},
        "signature": {"status": "unsigned", "note": "SHA256SUMS detects corruption only. Production must verify a trusted publisher "
                                                    "signature over manifest.json (see backend/docs/operations/model_deployment.md)."},
        "runtime_requirements": {"python_packages": ["numpy", "scipy", "onnxruntime (CPU) or onnxruntime-gpu (optional)", "safetensors (optional, canonical weights)"],
                                 "np_load": "allow_pickle=False"},
        "release_complete": status == "complete",
    })


write_manifest("draft")
print("draft release written:", sorted(p.name for p in RELEASE_DIR.iterdir()))

## 20. Validate artifacts, test inference, then mark the release complete

The draft release is read back **from disk** (not from memory) and validated: required files, dtypes, shapes, finiteness, sparse-matrix validity, item-index alignment, attention config vs Safetensors shapes, ONNX graph validity, ONNX vs PyTorch parity, CPU inference (unseen user, ablations, fallbacks, determinism), content vectors for a brand-new item, ALS fold-in, and agreement between the serving code and the batched evaluation. GPU parity runs only when a CUDA ONNX Runtime provider is genuinely loaded; the actual provider is recorded. Any failure raises and the release stays `draft`.

In [ ]:
with model_stage("artifact_validation", critical=True):
    VALIDATION_DRAFT = run_artifact_tests(RELEASE_DIR, strict_load=False)
    print(json.dumps(VALIDATION_DRAFT, indent=1, default=_json_default))
    write_manifest("complete")
    n_sums = ref.write_sha256sums(RELEASE_DIR)
    ref.verify_sha256sums(RELEASE_DIR)
    print(f"release marked complete; {n_sums} files checksummed and verified")

## 21. Benchmark CPU and GPU inference

* Attention is benchmarked through four backends where available: ONNX Runtime CPU, ONNX Runtime CUDA, PyTorch CPU, PyTorch CUDA. A backend is reported as GPU **only** if its session reports `CUDAExecutionProvider` as the active provider.
* Models are loaded once; inputs are converted per call, so host-to-device transfers are included in GPU timings (that is what a request actually pays).
* Batch sizes 1, 16, 128 users x 100/400 candidates x short/long histories. p50/p95/p99 latency, throughput, process RSS and GPU memory are recorded.
* End-to-end single-user latency is measured with the reference recommender loaded once, per stage.
* Small requests may be faster on CPU; the results decide, not an assumption.

In [ ]:
def bench_attention(release_dir):
    import onnxruntime
    cfg = json.loads((release_dir / "attention_config.json").read_text())
    L = cfg["max_history"]
    from safetensors.torch import load_file
    model_cpu = build_attention_model(cfg).eval()
    model_cpu.load_state_dict(load_file(str(release_dir / "attention.safetensors")))
    backends, notes = {}, {}
    rss_before = rss_mb()
    s_cpu = ref.make_ort_session(release_dir / "attention.onnx", use_gpu=False)
    backends["onnxruntime-cpu"] = (lambda h, c: ref.attention_scores_onnx(s_cpu, h, c), s_cpu.get_providers()[0], False)
    if torch.cuda.is_available() and "CUDAExecutionProvider" in onnxruntime.get_available_providers():
        g0 = gpu_mem_used_mb()
        s_gpu = ref.make_ort_session(release_dir / "attention.onnx", use_gpu=True)
        if s_gpu.get_providers()[0] == "CUDAExecutionProvider":
            backends["onnxruntime-cuda"] = (lambda h, c: ref.attention_scores_onnx(s_gpu, h, c), "CUDAExecutionProvider", True)
            notes["onnxruntime_cuda_session_gpu_mb"] = None if g0 is None else (gpu_mem_used_mb() or 0) - g0
        else:
            notes["onnxruntime-cuda"] = "NOT BENCHMARKED: session fell back to " + s_gpu.get_providers()[0]
    else:
        notes["onnxruntime-cuda"] = "unavailable (no CUDA device or no CUDA execution provider)"
    backends["pytorch-cpu"] = (lambda h, c: model_cpu(torch.as_tensor(h), torch.as_tensor(c)).numpy(), "cpu", False)
    if torch.cuda.is_available():
        m_gpu = build_attention_model(cfg).eval().to("cuda")
        m_gpu.load_state_dict(load_file(str(release_dir / "attention.safetensors")))
        backends["pytorch-cuda"] = (lambda h, c: m_gpu(torch.as_tensor(h).cuda(), torch.as_tensor(c).cuda()).float().cpu().numpy(), "cuda", True)
    rng = np.random.default_rng(SEED + 13)
    rows = []
    with torch.no_grad():
        for name, (fn, provider, on_gpu) in backends.items():
            for B in (1, 16, 128):
                for C in (100, 400):
                    for hist_len in (8, L):
                        h = ref.pad_histories([rng.integers(0, n_items, size=hist_len) for _ in range(B)], L)
                        c = rng.integers(1, n_items + 1, size=(B, C)).astype(np.int64)
                        n_iter = 40 if B <= 16 else 12
                        stats = time_calls(lambda: fn(h, c), n_iter, warmup=3, sync=on_gpu)
                        rows.append({"backend": name, "execution_provider": provider, "batch": B, "candidates": C, "history_len": hist_len,
                                     **stats, "users_per_sec": B / (stats["mean_ms"] / 1000.0),
                                     "candidates_per_sec": B * C / (stats["mean_ms"] / 1000.0)})
    notes["rss_mb_growth_during_attention_bench"] = rss_mb() - rss_before
    notes["gpu_memory_used_mb_after"] = gpu_mem_used_mb()
    if torch.cuda.is_available():
        notes["torch_cuda_max_memory_allocated_mb"] = torch.cuda.max_memory_allocated() / 1e6
    return pd.DataFrame(rows), notes


def bench_end_to_end(release_dir, use_gpu):
    rec = ref.CineRankRecommender(release_dir, use_gpu=use_gpu)
    idx = np.random.default_rng(SEED + 14).choice(len(EVAL_VAL.users), size=min(BENCH_USERS, len(EVAL_VAL.users)), replace=False)
    hists = [[(int(i), float(r)) for i, r in zip(EVAL_VAL.hist_items[u], EVAL_VAL.hist_ratings[u])] for u in idx]
    for h in hists[:3]:
        rec.recommend(h, k=20)
    stages = {}
    for h in hists:
        t = rec.recommend(h, k=20, return_timings=True)["timings_ms"]
        for k, v in t.items():
            stages.setdefault(k, []).append(v)
    out = {k: percentiles_ms(v) for k, v in stages.items()}
    out["attention_execution_provider"] = rec.attention_provider
    out["pop_topk_ms"] = time_calls(lambda: ref.topk_indices(rec.popularity, 20), 200)
    return out


def bench_als_gpu_scoring(release_dir):
    Y = np.load(release_dir / "als_item_factors.npy", allow_pickle=False)
    out = []
    Yg = torch.as_tensor(Y).cuda()
    for B in (1, 64, 512):
        X = np.random.default_rng(0).normal(size=(B, Y.shape[1])).astype(np.float32)
        cpu = time_calls(lambda: ref.topk_indices((X @ Y.T)[0], 20), 30)
        gpu = time_calls(lambda: torch.topk(torch.as_tensor(X).cuda() @ Yg.T, 20).indices.cpu().numpy(), 30, sync=True)
        out.append({"batch": B, "numpy_cpu": cpu, "torch_cuda_resident_item_factors": gpu})
    return out


BENCHMARKS = {"environment": ENVIRONMENT}
with model_stage("benchmark"):
    attn_df, attn_notes = bench_attention(RELEASE_DIR)
    BENCHMARKS["attention"] = {"rows": attn_df.to_dict("records"), "notes": attn_notes}
    display(attn_df.pivot_table(index=["batch", "candidates", "history_len"], columns="backend", values="p95_ms").round(2))
    BENCHMARKS["end_to_end_cpu"] = bench_end_to_end(RELEASE_DIR, use_gpu=False)
    if torch.cuda.is_available() and "CUDAExecutionProvider" in ENVIRONMENT["onnxruntime_available_providers"]:
        BENCHMARKS["end_to_end_gpu_attention"] = bench_end_to_end(RELEASE_DIR, use_gpu=True)
    if torch.cuda.is_available():
        BENCHMARKS["als_scoring_cpu_vs_gpu"] = bench_als_gpu_scoring(RELEASE_DIR)
    print(json.dumps({k: (v["p95_ms"] if isinstance(v, dict) and "p95_ms" in v else v) for k, v in BENCHMARKS["end_to_end_cpu"].items()}, indent=1, default=_json_default))
    print("attention notes:", json.dumps(attn_notes, indent=1, default=_json_default))
    LATENCY_P95 = {"popularity": BENCHMARKS["end_to_end_cpu"]["pop_topk_ms"]["p95_ms"]}
    for model_name in ("content", "itemknn", "als", "bpr", "attention"):
        s = BENCHMARKS["end_to_end_cpu"].get(f"{model_name}_ms")
        if s:
            LATENCY_P95[model_name] = s["p95_ms"]
    LATENCY_P95["hybrid"] = BENCHMARKS["end_to_end_cpu"]["total_ms"]["p95_ms"]

## 22. Final results tables (test split, generated from the evaluation above)

Latency is single-user, CPU, models already loaded (stage timings of the reference recommender; `hybrid` is the whole request). Nothing in these tables is illustrative; they are produced by the cells above.

In [ ]:
def df_to_md(df, floatfmt="{:.4f}"):
    cols = [str(df.index.name or "")] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for idx, row in df.iterrows():
        cells = [str(idx)] + [floatfmt.format(v) if isinstance(v, (float, np.floating)) and np.isfinite(v) else ("n/a" if isinstance(v, (float, np.floating)) else str(v)) for v in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


_summary = results_frame(test_reports_pool, ("recall@10", "ndcg@10", "coverage@10", "diversity@10", "novelty@10", "long_tail_exposure@10"))
_summary["latency_p95_ms (CPU, 1 user)"] = [LATENCY_P95.get(m.replace("_no_mmr", ""), np.nan) for m in _summary.index]
print("TEST, shared candidate pool (primary comparison; protocol B):")
display(_summary.round(4))
_full_summary = results_frame(test_reports_full)
print("TEST, full catalogue (protocol A):")
display(_full_summary.round(4))
print("Interpretation limits: one held-out positive per user; sampled MovieLens users; per-user temporal split; "
      "pool-restricted ranking caps every model at the pool recall above.")

## 23. Run directory (diagnostics), then package, extract and verify the bundle

`training-runs/<run_id>/` holds diagnostics and reproducibility records: `training-config.json`, `environment.json`, `metrics.json`, `evaluation.json`, `evaluation_report.md`, `attention_training.json`, `attention_final.safetensors` (final-step weights), and `reference_inference.py`. It is **not** a production inference release and contains no raw ratings, user ids or holdout rows (aggregate metrics only).

The bundle is built as `<release_id>.zip.partial`, extracted with the safe extractor into a fresh directory, checked against its checksums, loaded with the production loader, tested again, scanned for prohibited content, and only then renamed to its final name.

In [ ]:
with model_stage("package", critical=True):
    from safetensors.torch import save_file as _save

    EVALUATION = {
        "protocol": {"split": "per-user temporal leave-two-out on positives (validation = second-latest, test = latest)",
                     "eval_users": int(len(EVAL_USERS)), "users_per_cohort_cap": EVAL_USERS_PER_COHORT, "ks": list(KS),
                     "ties": "pessimistic", "map_equals_mrr": "one relevant item per user", "history_exclusion": "all prior events incl. low ratings",
                     "candidate_pool_per_source": CANDIDATE_POOL_PER_SOURCE, "generators": GENERATORS},
        "validation": {"full_catalogue": val_reports_full, "shared_pool": val_reports_pool, "pool_recall": VAL_POOL_RECALL},
        "test": {"full_catalogue": test_reports_full, "shared_pool": test_reports_pool, "pool_recall": TEST_POOL_RECALL},
        "hybrid": {"normalization": HYB_METHOD, "weights": HYB_WEIGHTS, "mmr_lambda": MMR_LAMBDA, "comparison_validation": HYBRID_COMPARISON,
                   "mmr_table_validation": mmr_table, "search": search_log},
        "latency_p95_ms_cpu_single_user": LATENCY_P95,
        "explicit_rating_diagnostic": EXPLICIT_DIAGNOSTIC,
    }
    METRICS = {"data_summary": DATA_SUMMARY, "dataset": DATASET_INFO, "stages": STAGES, "attention_training": ATTENTION_TRAINING,
               "selected_hyperparameters": SELECTED, "training_records": TRAINING_RECORDS, "artifact_validation": VALIDATION_DRAFT,
               "benchmarks": BENCHMARKS, "peak_rss_mb": peak_rss_mb(),
               "peak_gpu_memory_mb": (torch.cuda.max_memory_allocated() / 1e6) if torch.cuda.is_available() else None}
    write_json(RUN_DIR / "training-config.json", {n: globals()[n] for n in CONFIG_NAMES})
    write_json(RUN_DIR / "environment.json", {**ENVIRONMENT, "seed": SEED,
                                              "reproducibility_note": "seeds are fixed, but GPU kernels, multithreaded BLAS and library versions can change results across hardware"})
    write_json(RUN_DIR / "metrics.json", METRICS)
    write_json(RUN_DIR / "evaluation.json", EVALUATION)
    (RUN_DIR / "reference_inference.py").write_text(REFERENCE_INFERENCE_SRC, encoding="utf-8")
    _save({k: v.contiguous() for k, v in attn_final_state.items()}, str(RUN_DIR / "attention_final.safetensors"),
          metadata={"format": "pt", "note": "final-step weights; the release uses the best-validation checkpoint"})

    t = ATTENTION_TRAINING
    report = [f"# Evaluation report: {RELEASE_ID}", "",
              "Generated by the training notebook from the measured results of this run. No value below is illustrative.", "",
              "## Setup", f"- Users in cohort: {DATA_SUMMARY['n_users']:,} (sample of MovieLens; seed {SEED}); catalogue items: {n_items:,}",
              f"- Evaluation users: {len(EVAL_USERS):,}; split: per-user temporal leave-two-out on positives (rating >= {POSITIVE_RATING_THRESHOLD})",
              f"- Dataset archive SHA-256: `{DATASET_INFO['archive_sha256']}`", "",
              "## Attention training (compute budget, not quality)",
              f"- Wall clock incl. validation/logging: {t['elapsed_wall_clock_sec']:.1f}s of {t['time_budget_sec']:.0f}s (overrun {t['budget_overrun_sec']:.2f}s)",
              f"- Optimizer steps: {t['optimizer_steps']:,}; examples processed: {t['examples_processed']:,} of {t['training_examples_available']:,} available per epoch ({t['epochs_completed_fraction']:.2f} epochs)",
              f"- Precision/device: {t['precision']} / {t['device']}",
              f"- Best sampled-validation checkpoint: step {t['best_checkpoint']['step']}, NDCG@10 {t['best_checkpoint']['ndcg@10']:.4f} (1 positive vs 99 sampled negatives; optimistic, for checkpoint selection only)",
              "", "## Test, shared candidate pool (primary comparison)", df_to_md(_summary),
              "", "## Test, full catalogue", df_to_md(_full_summary),
              "", "## Pool recall (ceiling for every re-ranker)", f"- test union pool recall: {TEST_POOL_RECALL['union_pool_recall']:.4f}",
              "", "## Test NDCG@10 by history cohort (shared pool)",
              df_to_md(pd.DataFrame({m: {c: r['ndcg@10'] for c, r in rep['cohorts'].items()} for m, rep in test_reports_pool.items()}).T),
              "", "## Test Recall@10 by target popularity band (shared pool)",
              df_to_md(pd.DataFrame({m: {b: r['recall@10'] for b, r in rep['target_bands'].items()} for m, rep in test_reports_pool.items()}).T),
              "", "## Hybrid (validation)", f"- normalisation `{HYB_METHOD}`, weights {HYB_WEIGHTS}, MMR lambda {MMR_LAMBDA}",
              f"- comparison (validation NDCG@10, pool): `{json.dumps({k: round(v, 4) for k, v in HYBRID_COMPARISON.items() if not isinstance(v, dict)})}`",
              "", "## BPR gate", f"- production enabled: {BPR_PRODUCTION_ENABLED}", "",
              "## Limitations",
              "- One held-out positive per user; a sample of MovieLens users is not the CineRank population.",
              "- Per-user temporal split, not a global date cutoff; popularity and collaborative statistics can include other users' later behaviour.",
              "- Hyperparameter grids are small; hybrid weights come from a random search, not an optimiser.",
              "- Pool-restricted ranking caps re-rankers at the pool recall; sampled in-training validation is optimistic.",
              "- Offline metrics do not establish online recommendation quality; MovieLens ratings are not CineRank behaviour."]
    (RUN_DIR / "evaluation_report.md").write_text("\n".join(report) + "\n", encoding="utf-8")
    ref.write_sha256sums(RUN_DIR)

    # ----- zip -> safe extract -> verify -> prohibited-content scan -> rename ------------------------------------------
    ALLOWED_SUFFIXES = {".npy", ".npz", ".json", ".safetensors", ".onnx", ".md", ".py"}
    FORBIDDEN_FRAGMENTS = ("ratings", "holdout", "train_mat", "user_factors", "users", "catalogue", "movies", ".csv", ".pkl", ".pickle",
                           ".joblib", ".pt", ".pth", ".sql", ".sqlite", ".db", ".parquet", ".dump")
    prefixes = (f"backend/artifacts/models/releases/{RELEASE_ID}/", f"backend/artifacts/training-runs/{RUN_ID}/")
    tmp_zip = Path(str(BUNDLE_ZIP) + ".partial")
    if BUNDLE_ZIP.exists() or tmp_zip.exists():
        raise FileExistsError(f"refusing to overwrite {BUNDLE_ZIP}")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in sorted(BUILD_ROOT.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=p.relative_to(BUILD_ROOT).as_posix())
    with zipfile.ZipFile(tmp_zip) as zf:
        names = zf.namelist()
    for n in names:
        assert n.startswith(prefixes), f"unexpected path in bundle: {n}"
        base = n.rsplit("/", 1)[-1]
        assert base == "SHA256SUMS" or Path(base).suffix in ALLOWED_SUFFIXES, f"disallowed file type: {n}"
        assert not any(f in base.lower() for f in FORBIDDEN_FRAGMENTS), f"prohibited content in bundle: {n}"
    assert (RUN_DIR / "evaluation.json").stat().st_size < 10_000_000, "evaluation.json unexpectedly large"
    print(f"bundle contains {len(names)} files; none are prohibited")

    VERIFY_ROOT = WORKSPACE / "cinerank_verify" / RELEASE_ID
    if VERIFY_ROOT.exists():
        raise FileExistsError(VERIFY_ROOT)
    ref.safe_extract_zip(tmp_zip, VERIFY_ROOT)
    extracted_release = VERIFY_ROOT / "backend" / "artifacts" / "models" / "releases" / RELEASE_ID
    extracted_run = VERIFY_ROOT / "backend" / "artifacts" / "training-runs" / RUN_ID
    assert extracted_release.is_dir() and extracted_run.is_dir(), "bundle does not extract into backend/artifacts/{models/releases,training-runs}"
    ref.verify_sha256sums(extracted_release)
    ref.verify_sha256sums(extracted_run)
    VALIDATION_EXTRACTED = run_artifact_tests(extracted_release, strict_load=True)
    tmp_zip.replace(BUNDLE_ZIP)
    print("extracted bundle verified (checksums, production loader, CPU inference):")
    print(json.dumps({k: VALIDATION_EXTRACTED[k] for k in ("onnx_cpu_parity", "onnx_gpu_parity", "cpu_inference")}, indent=1, default=_json_default))
    print(f"\nbundle ready: {BUNDLE_ZIP} ({BUNDLE_ZIP.stat().st_size / 1e6:.1f} MB)")

## 24. Completion report

This cell prints what was actually executed in *this* run, keeping syntax validation, notebook execution, model training, artifact validation and inference testing separate.

In [ ]:
_t = ATTENTION_TRAINING
_gpu_tested = isinstance(VALIDATION_EXTRACTED.get("onnx_gpu_parity"), dict) and "actual_provider" in VALIDATION_EXTRACTED["onnx_gpu_parity"] \
    and VALIDATION_EXTRACTED["onnx_gpu_parity"]["actual_provider"] == "CUDAExecutionProvider"
report_lines = [
    ("notebook executed top to bottom in this runtime", True),
    ("stages", {k: v["status"] for k, v in STAGES.items()}),
    ("attention budget respected", f"{_t['elapsed_wall_clock_sec']:.1f}s of {_t['time_budget_sec']:.0f}s (overrun {_t['budget_overrun_sec']:.2f}s)"),
    ("attention optimizer steps", _t["optimizer_steps"]),
    ("artifact validation (draft + extracted bundle)", "passed"),
    ("ONNX vs PyTorch CPU max abs diff", VALIDATION_EXTRACTED["onnx_cpu_parity"]["max_abs_diff"]),
    ("CPU inference tested", True),
    ("GPU inference tested (ONNX Runtime CUDA actually active)", _gpu_tested),
    ("BPR production-enabled", BPR_PRODUCTION_ENABLED),
    ("manifest status", json.loads((extracted_release / "manifest.json").read_text())["status"]),
    ("signature", "UNSIGNED: sign manifest.json before distributing across machines"),
    ("bundle", str(BUNDLE_ZIP)),
]
for k, v in report_lines:
    print(f"{k:58s} {v}")

## 25. Download the bundle (optional Drive copy)

In [ ]:
try:
    from google.colab import files
    files.download(str(BUNDLE_ZIP))
except ImportError:
    print("Not running in Google Colab. Bundle is available at:", BUNDLE_ZIP)

SAVE_TO_GOOGLE_DRIVE = False
GOOGLE_DRIVE_DESTINATION = f"/content/drive/MyDrive/CineRank/{RELEASE_ID}.zip"
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path(GOOGLE_DRIVE_DESTINATION)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        raise FileExistsError(f"refusing to overwrite {dest}")
    shutil.copy2(BUNDLE_ZIP, dest)
    print("Copied bundle to:", dest)